In [1]:
import dataiku
import pandas as pd
import json

In [2]:
import requests
from typing import Literal
# from eval_harness.core.models import AdapterError

RETRYABLE_STATUS = {429, 500, 502, 503, 504}

def post_ecs(endpoint: str, headers: dict, payload: dict, timeout: int) -> dict:
    """POST to a ECS API endpoint and return parsed JSON.

    Raises:
        AdapterError(retryable=True): if status is 429 or 5xx
        requests.HTTPError: for other non-2xx responses
    """
    response = requests.post(endpoint, headers=headers, json=payload, timeout=timeout)
    if response.status_code in RETRYABLE_STATUS:
        raise f"ECS HTTP {response.status_code}: {response.text[:200]}"
    response.raise_for_status()
    return response.json()

In [3]:
# ECS API endpoints (ask the AIWA/ECS project team for these)
UPLOAD_ENDPOINT       = "https://api-ecs-services-dev.otsuka-us.com/ecs-utils/public/api/v1/ecs-utils/file-upload/run"
CRF_EXTRACT_ENDPOINT  = "https://api-ecs-services-dev.otsuka-us.com/ecs-utils/public/api/v1/ecs-utils/extract-crf/run"
GENERATE_REVIEW_TABLE = "https://api-ecs-services-dev.otsuka-us.com/ecs-utils/public/api/v1/ecs-utils/generate-review-table/run"
# SECTION_LIST_ENDPOINT = "https://api-plps-services-dev.otsuka-us.com/plps-utils/public/api/v1/plps-utils/get-section-list/run"
GENERATE_ENDPOINT     = "https://api-ecs-services-dev.otsuka-us.com/ecs-platfrom-agent/public/api/v1/ecs-platfrom-agent/ecs-generation/run"

# Bearer tokens (Dataiku → Settings → API keys, or project service account)
GENERATE_API_KEY = "UELNIsNSfnUCx2eglTEp88zRKEARqvCk"
UTILS_API_KEY    = "y9tRT98VIbDKD19gQ9G4wd8z7lw70uUY"

# Dataiku managed folder holding the protocol .pdf files
PROTOCOL_FOLDER = "raw_protocols"

## Upload API

In [4]:
import base64

# Put inside adapter
def _read_file_as_base64(protocol_filename: str, protocol_folder: str = "file_upload") -> str:
    """ Read protocol file as base64 """
    handler = dataiku.Folder(protocol_folder)
    with handler.get_download_stream(protocol_filename) as f:
        file_b64 = base64.b64encode(f.read()).decode("utf-8")
    return file_b64


b64 = _read_file_as_base64("405-201-00150 Annotated CRF__ v5.00 24 Jul 2024.pdf")
len(b64)

2157080

In [5]:
upload_payload = {
    "user_id": "eval_harness",
    "file_b64": b64,
    "file_name": "405-201-00150 Annotated CRF__ v5.00 24 Jul 2024.pdf"
}

upload_resp = post_ecs(
    endpoint=UPLOAD_ENDPOINT,
    headers={"Authorization": f"Bearer {UTILS_API_KEY}"},
    payload=upload_payload,
    timeout=300
)

In [6]:
print(json.dumps(upload_resp, indent=2))

{
  "response": {
    "message": "success",
    "crf_file_id": "bd82861a-f084-4859-8a85-86f195ffe855",
    "file_path": "/405-201-00150 Annotated CRF__ v5.00 24 Jul 2024.pdf"
  },
  "timing": {
    "preProcessing": 0,
    "wait": 70,
    "execution": 2969858,
    "functionInternal": 2961341
  },
  "apiContext": {
    "serviceId": "ecs-utils",
    "endpointId": "file-upload",
    "serviceGeneration": "v11"
  }
}


## Digitize API

In [7]:
# extract_payload = {
#     "file_path": upload_resp["response"]["file_path"],
#     "file_id": upload_resp["response"]["crf_file_id"],
#     "created_by": "eval_harness"
# }

# extract_resp = post_ecs(
#     endpoint=CRF_EXTRACT_ENDPOINT,
#     headers={"Authorization": f"Bearer {UTILS_API_KEY}"},
#     payload=extract_payload,
#     timeout=300
# )

# print(json.dumps(extract_resp, indent=2))

## Generate Table Review 

In [8]:
generate_table_resp = post_ecs(
    endpoint=GENERATE_REVIEW_TABLE,
    headers={"Authorization": f"Bearer {UTILS_API_KEY}"},
    payload={
        "file_id": upload_resp["response"]["crf_file_id"],
        "file_path": upload_resp["response"]["file_path"],
        "indication": "",
        "molecule": "SEP-380135",
        "ta": ""
    },
    timeout=300
)

print(json.dumps(generate_table_resp, indent=2))

{
  "response": {
    "result": []
  },
  "timing": {
    "preProcessing": 0,
    "wait": 33,
    "execution": 2887892,
    "functionInternal": 2887486
  },
  "apiContext": {
    "serviceId": "ecs-utils",
    "endpointId": "generate-review-table",
    "serviceGeneration": "v11"
  }
}


In [9]:
generate_table_resp = 
{
  "response": {
    "result": [
      {
        "validation_id": "",
        "ecs_id": "07708d45-2511-429b-921c-37454a6ec19d",
        "form_id": "",
        "form_name": "Date of Visit",
        "form_field_value": "Visit date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "SVSTDAT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2afb8dc4-fab6-4bd7-ab5e-ffce253d58e0",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Informed consent obtained?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6eadf2b1-b5b4-47aa-b115-99803687ddf0",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Informed consent date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2998ebb1-d1f8-4be0-9702-cb0d560e5d03",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Informed consent time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7874a16b-0e68-4b69-ac01-9153cca38dd8",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "404c5487-7cec-42fc-8928-0ec01f1d1ae7",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Informed consent version number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "587f4a47-7de9-4b95-9b1e-39fc752a29d2",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Standardized disposition term",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "555e5767-9cb8-4feb-bff5-fa23bdf72d0f",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Protocol version",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "909f048b-12ee-4fc1-b0a5-6a7d8182fa1f",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Amendment 1",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "339d637f-de63-46f4-8c19-449ec3dba01b",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Amendment 2",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "24269c6e-5453-4aa4-9ab7-6da00bd18563",
        "form_id": "",
        "form_name": "Informed Consent",
        "form_field_value": "Amendment 3",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSTIM",
          "DSDTTM",
          "DSSAMND",
          "DSDECOD_3",
          "PROTOCOL",
          "DSYN",
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "MVAL_DM023",
        "ecs_id": "0c88ec50-e459-42a1-83d6-b6206a766f1c",
        "form_name": "Demographics",
        "validation_logic": "(DM.ETHNIC is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['ETHNIC']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_DM027",
        "ecs_id": "884e9ba7-5025-4249-b55d-43fe873651d9",
        "form_name": "Demographics",
        "validation_logic": "(DM.COUNTRY is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['COUNTRY']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_DM017",
        "ecs_id": "24ccdee2-e2c3-41e6-8cd2-30ba4663190b",
        "form_name": "Demographics",
        "validation_logic": "(DM.AGE is not numeric)",
        "reasoning": "Field type must be numeric",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AGE']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for non-conformant data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_DM016",
        "ecs_id": "23b8c00a-01f5-4899-8a6d-e966284d0db0",
        "form_name": "Demographics",
        "validation_logic": "(DM.AGE is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AGE']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_DM019",
        "ecs_id": "c1fbd92c-87ab-4e6a-860e-f8ba31ea1176",
        "form_name": "Demographics",
        "validation_logic": "(DM.SEX is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['SEX']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "DYN_DM_SET_COUNTRY",
        "ecs_id": "ef5db180-023c-4108-95c5-d47828421b9a",
        "form_name": "Demographics",
        "validation_logic": "If AGE in Demographics IsPresent  then... AGE in Demographics IsPresent, and set site information: Country",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['AGE']",
        "action": "Set Datapoint",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00002"
      },
      {
        "validation_id": "",
        "ecs_id": "9c42e555-b437-433b-b932-0ad9fd54a7b6",
        "form_id": "",
        "form_name": "Inclusion/Exclusion Criteria",
        "form_field_value": "Did participant satisfy all Inclusion/Exclusion criteria?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "IEYN",
          "IECAT",
          "IEYN",
          "IETESTCD"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "27896c41-d700-470a-9f8e-576618aa5ac8",
        "form_id": "",
        "form_name": "Inclusion/Exclusion Criteria",
        "form_field_value": "If no, Inclusion/Exclusion category",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "IEYN",
          "IECAT",
          "IEYN",
          "IETESTCD"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ffed740c-51d3-4abb-b4b2-f8e8bbd25c84",
        "form_id": "",
        "form_name": "Inclusion/Exclusion Criteria",
        "form_field_value": "EXCLUSION",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "IEYN",
          "IECAT",
          "IEYN",
          "IETESTCD"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8279de2a-6377-48e9-9865-2bc728ec83c0",
        "form_id": "",
        "form_name": "Inclusion/Exclusion Criteria",
        "form_field_value": "Inclusion/Exclusion criterion Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "IEYN",
          "IECAT",
          "IEYN",
          "IETESTCD"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "MVAL_MH018",
        "ecs_id": "a8eeba78-9a10-4ebd-9a85-a3cc4606d28b",
        "form_name": "Medical and Surgical History",
        "validation_logic": "(MH.MHTERM is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['MHTERM']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_MH021",
        "ecs_id": "1648c3b6-a508-49a7-aec7-75e023e7363a",
        "form_name": "Medical and Surgical History",
        "validation_logic": "(MH.MHSTDAT is an invalid date)",
        "reasoning": "Field must not be an invalid date such as 31Feb2025",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['MHSTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for invalid date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_MH025",
        "ecs_id": "7b791a27-864e-40d2-b54a-04b4d0b0d5a5",
        "form_name": "Medical and Surgical History",
        "validation_logic": "(MH.MHENDAT is a future date)",
        "reasoning": "Field must not be an invalid date such as 31Feb2025",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['MHENDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for invalid date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_MH023",
        "ecs_id": "b540c5d5-4f29-484d-8586-d97fdf93bca1",
        "form_name": "Medical and Surgical History",
        "validation_logic": "(MH.MHENDAT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['MHENDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_MH024",
        "ecs_id": "9bce23fd-578f-4574-8877-92abbf07571f",
        "form_name": "Medical and Surgical History",
        "validation_logic": "(MH.MHENDAT is a future date)",
        "reasoning": "Field must not be a date in the future",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['MHENDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for future date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_MH027",
        "ecs_id": "6ec8949d-b387-432f-8194-deb16a4d51d1",
        "form_name": "Medical and Surgical History",
        "validation_logic": "(MH.MHENDAT < MH.MHSTDAT)",
        "reasoning": "end date can never be prior to start date",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['MHENDAT', 'MHSTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "End date is prior to start date. Please correct or clarify.",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_MH016",
        "ecs_id": "c157341b-d4ca-4392-93fd-14c5bf16af03",
        "form_name": "Medical and Surgical History",
        "validation_logic": "(MH.MHCAT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['MHCAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_MH019",
        "ecs_id": "0e5ed3e0-605c-4762-b238-0d0cb7528b58",
        "form_name": "Medical and Surgical History",
        "validation_logic": "(MH.MHSTDAT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['MHSTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_MH020",
        "ecs_id": "9b69a604-3769-4f62-ab78-d3a8ea943ada",
        "form_name": "Medical and Surgical History",
        "validation_logic": "(MH.MHSTDAT is a future date)",
        "reasoning": "Field must not be a date in the future",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['MHSTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for future date>",
        "path": "nan"
      },
      {
        "validation_id": "PE_TARGET003",
        "ecs_id": "a56b626e-f11c-4627-bc2e-7ef0a927a158",
        "form_name": "Physical Examination",
        "validation_logic": "Flag if PE_TARGET.PEDAT is not equal to SV.VISDAT, THEN fire query on PE_TARGET.PEDAT (Exam Date)",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['PEDAT', 'VISDAT']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "The Physical Examination- Target 'Exam Date' is not equal to visit date. Please review and update or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS017",
        "ecs_id": "9f00f254-1085-419d-a96f-194cf1c01f79",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS015",
        "ecs_id": "0bd16a1f-4630-4e44-887e-bedb672573f7",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS016",
        "ecs_id": "8fa45683-dca3-4b4f-833f-af145e477200",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND_LOG013",
        "ecs_id": "9f72c18b-307a-43ce-aa11-d5c29d928bb0",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND_LOG015",
        "ecs_id": "af6dd79a-ead6-4665-a3c7-31ea526467ca",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG016",
        "ecs_id": "97d79040-5f9c-40b1-a7e6-e7df436516ca",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG018",
        "ecs_id": "4660f51e-55c7-4a21-af56-671c6010b0fa",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0264b879-daa4-447d-a695-c0c0d49d93f2",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Query if standing position time is not at least 3 minutes (+59 sec window) after supine position time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0f7c71a8-52c8-4bf0-b7e3-2aa01edae5cd",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND005",
        "ecs_id": "8865576d-b8c7-4792-b230-cdc5fc7cb56e",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND007",
        "ecs_id": "982a060a-f224-4f12-8150-56a469450c96",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND006",
        "ecs_id": "189a39c0-2f22-466d-9bea-50f30e28b2e7",
        "form_name": "Vital Signs (Scr)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "5f9bbae4-35a1-4cd2-a983-9f26645197d2",
        "form_name": "Weight/Height/BMI",
        "validation_logic": "Item should not be selected if an end date is present for the substance use.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['BMI']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "TEMPLATE_WEIGHT",
        "ecs_id": "1a3ea5a3-33fd-4b13-b229-0ea4c0a4d860",
        "form_name": "Weight/Height/BMI",
        "validation_logic": "Value should = Y",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['WEIGHT']",
        "action": "Is Greater than or Equal to 50",
        "source": "Historical",
        "action_details": "Weight is out of range, please review.",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "23d46683-0438-4f3f-a98d-078eece91816",
        "form_name": "Electrocardiogram Screening",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "fd740ca4-c8ed-461b-9b38-f84d51e05439",
        "form_name": "Electrocardiogram Screening",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "e0ce35a0-e237-4302-a700-42f0b0ed55bb",
        "form_name": "Electrocardiogram Screening",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "ec08d46a-876b-411b-a227-b53e96510eec",
        "form_name": "Electrocardiogram Screening",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "",
        "ecs_id": "b61b2e5e-5b46-46f8-b4b9-9de5ac04c474",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8e7a5658-05f7-4607-9c6b-43d4b4fd22d7",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4d74ca98-1c98-4cd3-87f3-f4f3d4f7bb11",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "Were lab assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "bd27f9ff-965d-4a4e-8e83-5f415340415d",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4b30950b-52fc-440d-9470-4c43093fe6e6",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "eb36f521-b201-4f6c-b9c9-4c6bb8ce1d83",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a8372b49-ca69-4e17-ac8d-30dda0b79d6f",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "aeae0897-d8bb-4b83-9d55-b8c99a03f67a",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "Are the results clinically significant?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "73f57c18-869a-4360-be8a-735afd4f7a36",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "If clinically significant, specify1",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "118aeacf-3deb-4237-ab6b-2055cd6e35f1",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "If clinically significant, specify2",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4baccee4-d010-40fa-b036-950a9c1406fe",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "If clinically significant, specify3",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "bca94767-c72f-40a3-a4ce-b226f39c95be",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "If other, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4d8954c3-7f3e-4739-99fa-416ac3cc4d4d",
        "form_id": "",
        "form_name": "Laboratory Test Collections Screening",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "60e29163-3955-4125-ade6-1b9c4b39a977",
        "form_id": "",
        "form_name": "Laboratory Test_DOA",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_2",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7970df4a-b46b-4636-b188-a1a4c6c804f7",
        "form_id": "",
        "form_name": "Laboratory Test_DOA",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_2",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "038b624d-9697-4e92-afe3-178e3d521216",
        "form_id": "",
        "form_name": "Laboratory Test_DOA",
        "form_field_value": "Were lab assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_2",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "864f0f52-6596-49d5-b64a-cc6a49695194",
        "form_id": "",
        "form_name": "Laboratory Test_DOA",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_2",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "71e50674-f16b-4f04-b7d2-9a412bbc5a44",
        "form_id": "",
        "form_name": "Laboratory Test_DOA",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_2",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7a30fdb6-561a-42d6-a038-4a0bab12ef1b",
        "form_id": "",
        "form_name": "Laboratory Test_DOA",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_2",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8a8c0510-7324-41f8-805b-e587e805e7c7",
        "form_id": "",
        "form_name": "Laboratory Test_DOA",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_2",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "936b5914-66b3-48b2-b8e7-18257b3e237f",
        "form_id": "",
        "form_name": "Laboratory Test_DOA",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_2",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f358c6d5-8dee-45ae-b467-5afec41dd4e2",
        "form_id": "",
        "form_name": "Laboratory Test_PG",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "82c9d436-ac58-4861-9062-2a0294be22dd",
        "form_id": "",
        "form_name": "Laboratory Test_PG",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8de54173-d50f-40f9-a964-70a2d20929b0",
        "form_id": "",
        "form_name": "Laboratory Test_PG",
        "form_field_value": "Urine",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "066edc60-b610-4838-b74e-3f8f03c42f79",
        "form_id": "",
        "form_name": "Laboratory Test_PG",
        "form_field_value": "Were lab assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9b25bc39-48c1-4fbb-9b31-b86ba747c6d8",
        "form_id": "",
        "form_name": "Laboratory Test_PG",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8388cc65-1dff-4d0b-9330-3497e2553ea9",
        "form_id": "",
        "form_name": "Laboratory Test_PG",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cee8bbbd-1ff2-452a-ab31-0eae1b747a1e",
        "form_id": "",
        "form_name": "Laboratory Test_PG",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f685ca30-ac80-40aa-bb88-1545cbf9d07f",
        "form_id": "",
        "form_name": "Laboratory Test_PG",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0b3a72eb-b19a-4095-aa5a-4e82f779afd7",
        "form_id": "",
        "form_name": "Laboratory Test_PG",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "bc615d42-badc-4ed2-995a-1f8d72f12d17",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e70cfbfd-2214-41dc-996d-5401ed308ed9",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a40d0268-27c6-4200-8956-d33c91b6bb51",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "Were lab assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "3d149159-5a15-41e1-9695-8afb3c2dff0b",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "14771496-65ad-4fd4-a9f8-4ce6840bf042",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cff19141-4f6a-4a76-8972-71462b0ebc1b",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "74a62a02-8f34-4d22-9360-45b9f3acaf2b",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "01760bbd-92d2-4f31-8cbb-fc0c125ecfba",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "Are the results clinically significant?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "3f16728a-7d77-4d38-82fc-7df3da120d6d",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "If yes, specify CS",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7c7dab32-668a-43bc-86be-30e268b56f46",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "097ca9a4-a7af-4903-bb06-17ec3209d3e2",
        "form_id": "",
        "form_name": "Screening Outcome",
        "form_field_value": "Is participant a screen failure?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_1",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "efaad370-4a40-476d-b597-e126cfcafaeb",
        "form_id": "",
        "form_name": "Screening Outcome",
        "form_field_value": "Date screening completed/failed",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_1",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "879fec3e-e0c3-4d12-a473-7a7dc91d90f5",
        "form_id": "",
        "form_name": "Screening Outcome",
        "form_field_value": "Screening completed or reason for screening failure",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_1",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ff64dbf4-96d7-4fd6-b655-852fa16e34e1",
        "form_id": "",
        "form_name": "Screening Outcome",
        "form_field_value": "If discontinued for 'death', enter date of death",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_1",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "467ed4a9-8cf9-4991-adc0-a86c71e307d6",
        "form_id": "",
        "form_name": "Screening Outcome",
        "form_field_value": "If discontinued for 'protocol deviation', 'death' or 'other', Please specify 5",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_1",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "31770c95-74c6-4571-9390-b948551bb7ae",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Category of Questionnaire (Defaulted, Hidden)",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b8a7faa9-989b-4cb0-8468-87e2abafeb9d",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Collection Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "bd6cb93e-249f-4f65-864e-a90173843de9",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Collection Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5ccca426-04ad-4055-a3ab-171b8dbc240c",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime: Time He/She Felt Most Suicidal",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "176ce748-bccc-4f98-9c1e-25ae2cfcb260",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 6 Months",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "300ffdae-1f01-4edb-a264-d6f9af8d4019",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "If yes, describe:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8cd68e9e-6b1e-4a24-9c40-eeefd8d1bfb1",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Subcategory of Questionnaire (Defaulted, Hidden)",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "77d42027-92c6-4348-b962-e755262ee219",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime - Most Severe Ideation",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5faf89f0-595f-4740-8ab4-d6fdd8732bba",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Description of Ideation",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0ae672ed-2baa-4fff-a5e8-61fc40b96a1d",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 6 Months - Most Severe Ideation",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5857b7a1-e57d-4834-9e4c-6a7df83d6fdd",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Once a week",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "104c46df-1594-4249-914c-00fb282790da",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime - Actual Attempt",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cd45cb8f-1478-4e60-8115-24f94979062f",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime - Total # of Attempts",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "3033e615-13ad-4bab-af8c-e7d86cdc39fb",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 2 years - Actual Attempt",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "54be373b-e266-4c6a-b63d-20c387ecc47b",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 2 years - Total # of Attempts",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "897f7a95-a325-4183-80c3-11e12afd77eb",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime - Has participant engaged in Non-Suicidal Self-Injurious Behavior? Yes 54",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f682b516-b348-450e-b9b3-74ddfe851288",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 2 years - Has participant engaged in Non-Suicidal Self-Injurious Behavior? Yes 55",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1ccd9ac7-3868-4cab-b6cd-0a978e794a02",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime - Interrupted Attempt",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "415789b2-b996-4836-854b-1dc740f2fe9e",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime - Total # of interrupted",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "93463c2d-c22d-4318-bb10-1dcca43f596d",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 2 years - Interrupted Attempt",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "49dce1f5-1668-44eb-ab5a-f2596aadbbd7",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 2 years -Total # of interrupted",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "fb6367aa-f824-4aa8-adef-0aa36e71fbd5",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime - Aborted Attempt",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "77e20690-a442-4ca6-bb0a-2e36c53c0378",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime - Total # of aborted",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b93fa443-a140-49a9-8350-a5d7337718f5",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 2 years - Aborted Attempt",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2c2d3270-85ba-4a4e-aeba-9be3f774fe1c",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 2 years - Total # of aborted",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e7e8a70b-231d-42d9-803f-8c446f6c72db",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Lifetime",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2bd7a9e2-d000-4965-be1a-2621309a810a",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Past 2 years",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "db8b5c4c-97a1-4d49-a77f-479d45230838",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Most Recent Attempt Date:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "90e287a1-74d9-461a-8221-3d960dee2907",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Most Recent Attempt Actual Lethality/Medical Damage: No physical damage or very minor 77",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "fdf0f322-8af0-4b84-b205-dd591f7880e7",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Most Recent Attempt Potential Lethality: Only Answer if Actual Lethality=0 Behavior not likely to result in injury 78",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "60142fe1-a197-48b9-8f0b-608b2a64993c",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Most Lethal Attempt Date:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "665a9917-e9e7-4fed-9840-19b2105b3673",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Most Lethal Attempt Actual Lethality/Medical Damage: No physical damage or very minor 81",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4541995a-fc5f-4ba9-94d3-3f7b12ee98b1",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "physical damage",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cb8bce3a-b923-462b-91b1-a840b46643fd",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Most Lethal Attempt Potential Lethality: Only Answer if Actual Lethality=0 Behavior not likely to result in injury 82",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0db4ecc7-ce35-4f56-a98f-da1a25718800",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Initial/First Attempt Date:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f790d9ca-a0f9-4b7e-8224-5b95fed317f9",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Initial/First Attempt Actual Lethality/Medical Damage: No physical damage or very minor 85",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4a483edf-ffaa-4e1d-9e33-653a11665c35",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening",
        "form_field_value": "Initial/First Attempt Potential Lethality: Only Answer if Actual Lethality=0 Behavior not likely to result in injury 86",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSSCAT5",
          "QSDAT",
          "QSTIM",
          "CSS0401A",
          "CSS0401B",
          "CSS0401C",
          "QSSCAT6",
          "CSS0406A",
          "CSS0406B",
          "CSS0406C",
          "CSS0407A",
          "CSS0412A",
          "CSS0413A",
          "CSS0412B",
          "CSS0413B",
          "CSS0414A",
          "CSS0414B",
          "CSS0415A",
          "CSS0416A",
          "CSS0415B",
          "CSS0416B",
          "CSS0417A",
          "CSS0418A",
          "CSS0417B",
          "CSS0418B",
          "CSS0419A",
          "CSS0419B",
          "CSS0421A",
          "CSS0421B",
          "CSS0421C",
          "CSS0422A",
          "CSS0422B",
          "CSS0422B",
          "CSS0422C",
          "CSS0423A",
          "CSS0423B",
          "CSS0423C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "VS017",
        "ecs_id": "9f00f254-1085-419d-a96f-194cf1c01f79",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS015",
        "ecs_id": "0bd16a1f-4630-4e44-887e-bedb672573f7",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS016",
        "ecs_id": "8fa45683-dca3-4b4f-833f-af145e477200",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND_LOG013",
        "ecs_id": "9f72c18b-307a-43ce-aa11-d5c29d928bb0",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND_LOG015",
        "ecs_id": "af6dd79a-ead6-4665-a3c7-31ea526467ca",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG016",
        "ecs_id": "97d79040-5f9c-40b1-a7e6-e7df436516ca",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG018",
        "ecs_id": "4660f51e-55c7-4a21-af56-671c6010b0fa",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0264b879-daa4-447d-a695-c0c0d49d93f2",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Query if standing position time is not at least 3 minutes (+59 sec window) after supine position time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0f7c71a8-52c8-4bf0-b7e3-2aa01edae5cd",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND005",
        "ecs_id": "8865576d-b8c7-4792-b230-cdc5fc7cb56e",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND007",
        "ecs_id": "982a060a-f224-4f12-8150-56a469450c96",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND006",
        "ecs_id": "189a39c0-2f22-466d-9bea-50f30e28b2e7",
        "form_name": "Vital Signs (DM1)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "5f9bbae4-35a1-4cd2-a983-9f26645197d2",
        "form_name": "Weight/BMI",
        "validation_logic": "Item should not be selected if an end date is present for the substance use.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['BMI']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "TEMPLATE_WEIGHT",
        "ecs_id": "1a3ea5a3-33fd-4b13-b229-0ea4c0a4d860",
        "form_name": "Weight/BMI",
        "validation_logic": "Value should = Y",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['WEIGHT']",
        "action": "Is Greater than or Equal to 50",
        "source": "Historical",
        "action_details": "Weight is out of range, please review.",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "23d46683-0438-4f3f-a98d-078eece91816",
        "form_name": "Triplicate Electrocardiogram (DM1)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "fd740ca4-c8ed-461b-9b38-f84d51e05439",
        "form_name": "Triplicate Electrocardiogram (DM1)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "e0ce35a0-e237-4302-a700-42f0b0ed55bb",
        "form_name": "Triplicate Electrocardiogram (DM1)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "ec08d46a-876b-411b-a227-b53e96510eec",
        "form_name": "Triplicate Electrocardiogram (DM1)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "",
        "ecs_id": "a6335e97-3a8b-448b-9556-b528ceec152a",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4da3844f-6143-4d77-b127-5ef10c1140b4",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "21412c74-f4a1-42f2-bf33-bbf24886ecd7",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "Were lab assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b4477cbd-cff3-45a2-a19e-b36626c72e99",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "78e8af5f-7386-4daa-b811-a3dade98e0e5",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "eeebe120-a033-4ae1-9311-5a3e6ed08582",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c85987b8-fd33-48b9-859c-5d2ac8740a07",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a0ecb507-98b4-4101-b5a3-0694a9fdf412",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "Are the results clinically significant?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "307048cf-d49e-4219-b72f-7b98f2026952",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "If clinically significant, specify1",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "15ea55e2-c454-4ffe-a41f-ac7a5062a045",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "If clinically significant, specify2",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c4083e2b-6441-4258-9f0c-bbd2f664c3a7",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "If clinically significant, specify3",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "43b6893b-8eba-4b93-8e16-ee76a9da5173",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "If other, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e1bed744-fafb-428f-b538-c21d3cac480f",
        "form_id": "",
        "form_name": "Laboratory Test Collections DM1",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "96cb583b-790c-4d15-a396-71e875c4ac50",
        "form_id": "",
        "form_name": "Screening DM1 Outcome",
        "form_field_value": "Is participant a screen failure?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_5",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5700bd70-1c91-4aef-9c64-7fb7857f2b1d",
        "form_id": "",
        "form_name": "Screening DM1 Outcome",
        "form_field_value": "Date screening completed/failed",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_5",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "07bc7c99-324e-4e5e-8868-4a5fbdd0ee0c",
        "form_id": "",
        "form_name": "Screening DM1 Outcome",
        "form_field_value": "Screening completed or reason for screening failure",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_5",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "85f91b85-e777-4869-813d-51e3ace7f619",
        "form_id": "",
        "form_name": "Screening DM1 Outcome",
        "form_field_value": "If discontinued for 'death', enter date of death",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_5",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a13aec58-377b-451f-9df5-8ebaa5f058c3",
        "form_id": "",
        "form_name": "Screening DM1 Outcome",
        "form_field_value": "If discontinued for 'protocol deviation', 'death' or 'other', Please specify 5",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_5",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "76e645de-e008-4f46-b778-db1876d1c103",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Was C-SSRS performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "330a67f4-0571-4bd7-b4fa-30ae7fc31390",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "38a1d136-5d85-4655-84a1-2eeb870c9dcd",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5f33458f-5e0d-4b8c-9dd0-dec3852f8d72",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Collection Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "05476753-7fc3-4720-bea0-5337045ae454",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "SUICIDAL IDEATION 5",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5e0b0d99-c665-4e22-8007-c22d25131fc5",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Wish to be Dead",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "74810612-12d5-45e0-99ae-e27cf38a4ba6",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "If yes, describe",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5e8fdc58-140f-4f78-a1c2-f12c8f7fe6ee",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Non-Specific Active Suicidal Thoughts",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "aae0dea7-2569-4198-8943-98d0d34dc3fb",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Active Suicidal Ideation with Any Methods (Not Plan) without Intent to Act Y 10",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "02d8a24c-1f40-4fc3-840b-0accf694eddd",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Active Suicidal Ideation with Some Intent to Act, without Specific Plan Y 12",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5aa532bc-2110-4ed2-bdaa-339c86336168",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Active Suicidal Ideation with Specific Plan and Intent",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "988bfa75-5807-4def-a236-3432784a14a4",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Most Severe Ideation Type # (1-5)",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "024191df-6d3e-46a5-b5e5-91cf9c3dad95",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Description of Ideation",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "32eb5cba-15c6-4f99-a402-c8704b53ae46",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Frequency:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "969aeb1a-dd28-4a37-8089-cbbd1dbfc719",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Once a week",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e71ee15a-a08c-49a2-be4d-7ae4f69a2b39",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Duration",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7e0065fb-a492-40c7-980e-7522b97cb396",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Controllability",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5a2c07d5-ee64-4bc7-b279-bc06744419f9",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Deterrents",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "83829e15-20ad-47a7-88dd-1925ce339025",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Reason for Ideation:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b67ec662-4fc4-4869-b070-3cb97c791452",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Actual Attempt:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "3e0eee76-262c-4d42-b512-54a4cbe23bf1",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Total # of attempts",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "00a09ca4-bc7e-41d8-a5cd-d132c5483a77",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Has study participant engaged in Non-Suicidal Self-Injurious Behavior? Y",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c7ffc3fb-5555-4aea-8241-fc5f1fe6e585",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Interrupted Attempt:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b6e2ca49-2d40-4bb5-b9f3-d045d8a2fdbc",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Total # of interrupted",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e9c5aa88-0259-44bd-9c52-87a1773af294",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Aborted Attempt:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9f7f5d5f-1e58-4466-b8e0-27dd6f0d74bb",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Total # of aborted",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "aee5bbe8-5ec5-4313-bfe5-714ddee0bbe0",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Preparatory Acts or Behavior:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c5c85219-d79e-4db1-a7e5-a825156d452a",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Suicidal behavior was present during the assessment period?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c2fd7b07-d615-4002-b257-1ca82ec3aaf6",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Completed Suicide:",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "523a6960-5ed4-4702-9b5d-9c90498ba562",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Most Lethal Attempt Date (dd MMM yyyy)",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7b627651-d807-4985-af5d-d648ccd17953",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Actual Lethality/Medical Damage Code",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "058149f7-fcf0-4020-93f7-646bbc224d94",
        "form_id": "",
        "form_name": "Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit",
        "form_field_value": "Potential Lethality: Only Answer if Actual Lethality=0 Behavior not likely to result in injury 43",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "QSYN",
          "QSREASND",
          "QSDAT",
          "QSTIM",
          "QSSCAT1",
          "CSS0201",
          "CSS0201A",
          "CSS0202",
          "CSS0203",
          "CSS0204",
          "CSS0205",
          "CSS0206",
          "CSS0206A",
          "CSS0207",
          "CSS0207",
          "CSS0208",
          "CSS0209",
          "CSS0210",
          "CSS0211",
          "CSS0212",
          "CSS0213",
          "CSS0214",
          "CSS0215",
          "CSS0216",
          "CSS0217",
          "CSS0218",
          "CSS0219",
          "CSS0220",
          "CSS0221",
          "CSS0222A",
          "CSS0222B",
          "CSS0222C"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e6527199-ade9-48a7-9593-81384f67fa1a",
        "form_id": "",
        "form_name": "Randomization",
        "form_field_value": "Was the participant randomized?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSRNDNO",
          "DSTREAT",
          "DSARM",
          "DSDECOD_4"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5e6ffe78-1379-417e-951e-e89b252311d9",
        "form_id": "",
        "form_name": "Randomization",
        "form_field_value": "Date of Randomization/Enrollment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSRNDNO",
          "DSTREAT",
          "DSARM",
          "DSDECOD_4"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "eb9468f1-82c2-4305-9dc3-be04291f8493",
        "form_id": "",
        "form_name": "Randomization",
        "form_field_value": "Randomization number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSRNDNO",
          "DSTREAT",
          "DSARM",
          "DSDECOD_4"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "3693cce1-1bf5-4e81-8ab5-5655acda1f3c",
        "form_id": "",
        "form_name": "Randomization",
        "form_field_value": "Treatment Sequence",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSRNDNO",
          "DSTREAT",
          "DSARM",
          "DSDECOD_4"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0d66f89b-c25e-4d24-9dd6-e2c7744635e9",
        "form_id": "",
        "form_name": "Randomization",
        "form_field_value": "Arm",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSRNDNO",
          "DSTREAT",
          "DSARM",
          "DSDECOD_4"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ba0c19d8-6aba-475b-94b0-2a3f026350eb",
        "form_id": "",
        "form_name": "Randomization",
        "form_field_value": "Standardized disposition term",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSRNDNO",
          "DSTREAT",
          "DSARM",
          "DSDECOD_4"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5111d014-2ae9-4500-a284-805cc45f3823",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Did participant fast a minimum of 10 hours before dosing?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9c086516-04ec-4db7-99b1-77d19fdcd77c",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Was the Medication Compliance assessment by Mouth Check complete?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1f2b95a4-ba0a-4c8b-b810-72647561113d",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "If no specify reason",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f6e7afc9-3e43-4ce7-bfee-acb727a7b64d",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Name of actual treatment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1255d100-3622-4cf3-9414-1be188ba51e3",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Was dose given per protocol?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2457af6d-3bcc-4561-b59b-22ba29eb06ad",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "fcacfb35-1ee4-496e-9716-6f1b36409dd3",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Dosage form",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "d4fcb7ec-8ed8-4b19-a17c-864bd0fe5b33",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Dose Strength",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "971608d4-2b1d-48ae-9689-bf9126872139",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Strength Unit",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cf2e9bf9-9627-41bf-b670-d71bdfd418ba",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Number of Units administered",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b6c6c789-041b-4d29-a0f4-e43d4b4aeb4f",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Total Dose Administered in the morning",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "67efd6e7-c6ba-41d7-a8da-0757b2d262ba",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Unit",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6ed0f759-87ed-4e9e-9b4c-be210f8e6858",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Was the capsule sprinkled on the applesauce?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "80ad23e2-ff8a-42a2-9a77-b0fa022b6c4e",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Was the capsule sprinkled onto the yogurt?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "71f15cbe-44ee-4630-b003-fd7c9651f5a3",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Was the capsule sprinkled into orange juice?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9491e61f-1cbd-4edd-8166-57b8de696678",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Was the Participant refrained from drinking water within 1 hour of Dosing? Y 16",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "116e58f4-dafe-4cbc-8d78-ecb132fc2cb1",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Was the Dose administered with 240 ml of room temperature still water? Y",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5341ff94-1d35-4310-a6a9-1a69b1cd5349",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Route",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cef1895c-42b6-4054-a864-7f1f0dff494d",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7ffc986a-32e7-4aed-8852-c059eb6addfe",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ea50cb9f-0cee-4cea-807b-f145d490a9e5",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Derived Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f8b0b845-1041-441e-8621-0829d6011205",
        "form_id": "",
        "form_name": "Morning Dosing",
        "form_field_value": "Was afternoon dose administered?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXFAST",
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM",
          "EXAFTERYN"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "94ac307d-ccbe-47b6-8e60-0c0477fea1cf",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Was the Medication Compliance assessment by Mouth Check complete?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7acf6823-8610-4a35-9f52-ffb0d5ecae59",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "If no specify reason",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ab786287-a93a-4df0-8a4b-d9a9cdb4c302",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Name of actual treatment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "767ab29d-50c0-4640-a0bf-384a44b391e3",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Was dose given per protocol?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1cfe9a64-4a3d-47b9-af38-f42b499232e8",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "263ac60d-b3f8-49c9-81fd-9c58029eedbb",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Dosage form",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b5884376-58c4-4ed0-943a-bc61be1f99b4",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Dose Strength",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4cc0b945-a410-4826-a4d9-cbf95aa3a68f",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Strength Unit",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c814f6b1-820a-4fde-a7a9-0731020c6625",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Number of Units administered",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "42956483-007e-4af8-ab25-120308372a8b",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Total Dose Administered in the afternoon",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c1aba1de-183e-4f26-addd-7af99a32e536",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Unit",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "60d64430-f70f-4037-8359-cece782e00f3",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Was the capsule sprinkled on the applesauce?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "538bd8b6-2454-4465-9f04-ec5a186e691c",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Was the capsule sprinkled onto the yogurt?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "89773c5d-b120-45bd-a4ea-01acf07263eb",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Was the capsule sprinkled into orange juice?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f5c6596c-d9b8-4f4a-bd58-e310354fa996",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Was the Participant refrained from drinking water within 1 hour of Dosing? Y 15",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "12b861ce-29bb-44cd-ab1e-a6f6eac40817",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Was the Dose administered with 240 ml of room temperature still water? Y",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a3724291-9668-41e3-ace3-b613bde04043",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Route",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6f4ae479-d415-4edc-a204-2ec8665ba9a5",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "fb72f8ad-c3fd-4e38-9ca2-c6ae56b51501",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8bbfe7f6-10f1-4fc8-9a0a-0b668d41ba2b",
        "form_id": "",
        "form_name": "Afternoon Dosing",
        "form_field_value": "Derived Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "EXYN",
          "EXMC",
          "EXTRT",
          "EXADJYN",
          "EXADJ",
          "EXDOSFRM",
          "EXDOSESTR",
          "EXDOSSTUN",
          "EXUNITS",
          "EXDOSE",
          "EXDOSU",
          "EXCAPYN",
          "EXYOGYN",
          "EXOJYN",
          "EXDKYN",
          "EXVOLYN",
          "EXROUTE",
          "EXSTDAT",
          "EXSTTIM",
          "EXSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "VS017",
        "ecs_id": "9f00f254-1085-419d-a96f-194cf1c01f79",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS015",
        "ecs_id": "0bd16a1f-4630-4e44-887e-bedb672573f7",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS016",
        "ecs_id": "8fa45683-dca3-4b4f-833f-af145e477200",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND_LOG013",
        "ecs_id": "9f72c18b-307a-43ce-aa11-d5c29d928bb0",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND_LOG015",
        "ecs_id": "af6dd79a-ead6-4665-a3c7-31ea526467ca",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG016",
        "ecs_id": "97d79040-5f9c-40b1-a7e6-e7df436516ca",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG018",
        "ecs_id": "4660f51e-55c7-4a21-af56-671c6010b0fa",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0264b879-daa4-447d-a695-c0c0d49d93f2",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Query if standing position time is not at least 3 minutes (+59 sec window) after supine position time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0f7c71a8-52c8-4bf0-b7e3-2aa01edae5cd",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND005",
        "ecs_id": "8865576d-b8c7-4792-b230-cdc5fc7cb56e",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND007",
        "ecs_id": "982a060a-f224-4f12-8150-56a469450c96",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND006",
        "ecs_id": "189a39c0-2f22-466d-9bea-50f30e28b2e7",
        "form_name": "Vital Signs (D1)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "",
        "ecs_id": "1aa6dd6a-5ee6-4a3f-883e-646b32e7d752",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7985677f-ba2d-4b74-adb9-167b5bba643a",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "58f33534-c3e9-429b-aaae-b56a22fd9cc4",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e0ded6be-4e3a-460d-b474-18b1b9d80085",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "66cb0f69-77c4-42e1-817c-44de8adc2241",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2354e0f3-e241-4734-a773-e03f0fb56098",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9ca46d1b-5c45-4d79-9727-577d170ee6b1",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "d61619a8-750d-4e44-83d1-941a3cc67c57",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6b8aa91b-5045-4799-ae1c-1205c442e01d",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e88fc768-24dd-4553-87a3-a215dcf116b1",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "54f5631f-3493-4b99-9cfd-3bf3b4eb358f",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "06ecc79c-8385-4abf-91d4-3587f90edb5b",
        "form_id": "",
        "form_name": "Pharmacokinetics (D1)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b80f4af5-d873-46c2-9bab-2f33649a7fd1",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8680d788-2f70-4841-945c-34f9564d13c3",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e647e90a-eec0-4e63-9b6f-d5395af28a1e",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9ec1011e-3e52-464b-a665-82618e8c8010",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "03419471-07c8-4144-a410-285de0a60401",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1ebb6d10-f79a-4ccf-8b11-e334a8271b0d",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ccf6a676-640f-473d-9867-f005ccf76f6b",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cd5381e4-7f00-4213-9f58-9db55fd4a890",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1f940281-6056-4737-b9d0-0a55e6655e1f",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7a732afa-4c8c-4d0f-b293-b9bab130d663",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0f33a163-7bd9-4c8e-895e-80545815a80f",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "030879d6-f5ca-4169-87a0-1fe81d14b767",
        "form_id": "",
        "form_name": "Pharmacokinetics (D2)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e85dd5bd-1ea1-439d-a118-22e16f19ff4d",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6013f239-55d9-4ab3-9d36-e3e08088b466",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "bf2f999e-957b-4c82-9f74-44b33278ea8e",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5c74ee0d-4624-4ade-9abd-26f121527a36",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ba935961-36bb-4494-8fb1-2f423e9875ab",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ec467f70-5c09-4a83-a2a8-3384d8ca9c81",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "16996f7d-e61b-4e87-9045-7356a403f037",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f23c939f-ef47-485c-bb1d-d01421706f00",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b2f7e596-84fe-4514-a7b3-ed27f13441dd",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0f72405c-8ca9-47d2-9eb6-5d6e013a17e3",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "37fb97ef-9dd4-4707-8bd3-d325da4d2ad5",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "15e8817f-5265-4c8c-acfe-73df20f36044",
        "form_id": "",
        "form_name": "Pharmacokinetics (D3)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "VS017",
        "ecs_id": "9f00f254-1085-419d-a96f-194cf1c01f79",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS015",
        "ecs_id": "0bd16a1f-4630-4e44-887e-bedb672573f7",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS016",
        "ecs_id": "8fa45683-dca3-4b4f-833f-af145e477200",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND_LOG013",
        "ecs_id": "9f72c18b-307a-43ce-aa11-d5c29d928bb0",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND_LOG015",
        "ecs_id": "af6dd79a-ead6-4665-a3c7-31ea526467ca",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG016",
        "ecs_id": "97d79040-5f9c-40b1-a7e6-e7df436516ca",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG018",
        "ecs_id": "4660f51e-55c7-4a21-af56-671c6010b0fa",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0264b879-daa4-447d-a695-c0c0d49d93f2",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Query if standing position time is not at least 3 minutes (+59 sec window) after supine position time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0f7c71a8-52c8-4bf0-b7e3-2aa01edae5cd",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND005",
        "ecs_id": "8865576d-b8c7-4792-b230-cdc5fc7cb56e",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND007",
        "ecs_id": "982a060a-f224-4f12-8150-56a469450c96",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND006",
        "ecs_id": "189a39c0-2f22-466d-9bea-50f30e28b2e7",
        "form_name": "Vital Signs (D4)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "",
        "ecs_id": "a3cdece9-1d1e-4f1c-8330-2278e4227453",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "18ebece4-cdde-47a6-86b0-a0a0fdb34271",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c73319ce-d5cb-49a7-817b-88efd04bf7a0",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4ee540ad-f381-4df1-b537-06e1c86b6fe4",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ea8aa3fa-a46d-4b3f-a2cf-279ee5aad034",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0761c865-8299-4390-aa49-bb16dfb3d72b",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c06e969b-37db-45a0-b044-1eeae37c2793",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "eeea9d98-cfa9-4e00-a24b-be37b6b39aff",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0c9c6e51-546e-4adc-8211-658c23029c5c",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "18dbbb74-5f36-47c4-ba64-e5856fdfc3e2",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2bb6aea8-63bd-4048-9fc3-fa75195613fb",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "d564404d-320c-44fa-996c-938cd6abc54e",
        "form_id": "",
        "form_name": "Pharmacokinetics (D4)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f3b8e802-37e8-473d-9076-1b2493a1ffea",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5d3667b9-6105-47e3-9187-3ed77171c6da",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c6d980bc-46da-4bbe-8e96-1c242f88f2cc",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8b8dac5e-7861-490b-bbbe-cb434b93c5d7",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ed0019fd-3879-460b-af9c-273d74f1e4f7",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1097567c-9ccc-40ee-b6f8-cf3802edb766",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "31452242-8046-4c27-96b8-67f689da981a",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2e3c55a5-769f-438d-8e96-d38b2aa0c653",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "045a783b-130f-408e-90ed-fbf1b80004a7",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4e69d7c7-6aa7-45c5-9c8b-538b174a0b56",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f4e0951a-80ff-4815-9fa4-27110a7c8976",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5746c2b0-7169-4623-929e-1f4fdcf470bc",
        "form_id": "",
        "form_name": "Pharmacokinetics (D5)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9ce3b525-067c-45bd-a7da-64b8c42447b1",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "24a53122-4872-48d8-b881-d46089cd74ec",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "48249451-49d8-4d75-92d3-856d1f3ae543",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "95dfb60e-9729-4c9d-896f-3750f6838c23",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8c6b505f-d37d-4ef7-81ea-54f87ae55d04",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "839f42f5-e776-4541-a717-dc97ffe0d5c4",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "d416d325-bb61-4d00-84a3-c2938b5164cc",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e4df1015-7ede-4252-bea4-2bd0240535e0",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0c60e351-0c35-40d2-8083-6ecf21e5d01c",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "69b4ea04-9691-4f9d-8d5d-99a9c619b862",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "53c20318-e6b7-4c0a-a089-64c0cfb65225",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "186e1065-90f5-4c9f-99b1-ebda53d2e8c8",
        "form_id": "",
        "form_name": "Pharmacokinetics (D6)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "VS017",
        "ecs_id": "9f00f254-1085-419d-a96f-194cf1c01f79",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS015",
        "ecs_id": "0bd16a1f-4630-4e44-887e-bedb672573f7",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS016",
        "ecs_id": "8fa45683-dca3-4b4f-833f-af145e477200",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND_LOG013",
        "ecs_id": "9f72c18b-307a-43ce-aa11-d5c29d928bb0",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND_LOG015",
        "ecs_id": "af6dd79a-ead6-4665-a3c7-31ea526467ca",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG016",
        "ecs_id": "97d79040-5f9c-40b1-a7e6-e7df436516ca",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG018",
        "ecs_id": "4660f51e-55c7-4a21-af56-671c6010b0fa",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0264b879-daa4-447d-a695-c0c0d49d93f2",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Query if standing position time is not at least 3 minutes (+59 sec window) after supine position time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0f7c71a8-52c8-4bf0-b7e3-2aa01edae5cd",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND005",
        "ecs_id": "8865576d-b8c7-4792-b230-cdc5fc7cb56e",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND007",
        "ecs_id": "982a060a-f224-4f12-8150-56a469450c96",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND006",
        "ecs_id": "189a39c0-2f22-466d-9bea-50f30e28b2e7",
        "form_name": "Vital Signs (D7)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "",
        "ecs_id": "9ac5d731-51fc-46b7-9005-1519a2b3ed21",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "65cee05a-69c0-441c-a49d-9a63fd0b3dcc",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "20a30796-b54c-457b-bc18-23fb4f62dc37",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0b2009cd-488e-49c7-bdbb-8bd94215b952",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7cfd8bd4-b52e-49ac-abaf-04994b83e26f",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b6d31da6-bd91-488d-895e-f9089126f053",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0bd18b4b-8cac-4ef9-9bd8-bd05d3351dfe",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5a9b1bfa-49bc-4f31-8fa5-6adabb1e9ff7",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "3b80533b-e31d-4794-a3e3-e23b8e029b4d",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b09784b7-20f6-420c-81bc-b8a9655f63a0",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "39b1f998-2ad6-4648-83fa-32fa63f5c600",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "608a8af0-bb57-4eb6-b6d8-cb314fe195be",
        "form_id": "",
        "form_name": "Pharmacokinetics (D7)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c03d91a3-81b7-4732-8cc3-76babaf25b23",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "118eaa08-1ad0-490a-aa5c-25aed3c559f3",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "bb069285-d4b8-4423-8e2f-510b8f87d750",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "fdacf524-2571-4911-83cc-320fc95f84b3",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5eafd49c-072e-417a-a7ab-f42f67bb758d",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f3712299-0b27-490b-b886-753adb21e66a",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e223c411-f185-412f-aa46-4ce34c2d0115",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "65f53df4-43ce-4a96-b2c5-ca4d52ea8839",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "85fb8acc-ac31-438d-84f0-98a32919a725",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ee4bc96e-45fd-424d-bacb-b7a92e75cf3e",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6842658f-871e-4ee4-87d8-388a1b9bb31d",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f2137966-e961-45e2-85c1-566f34b4999d",
        "form_id": "",
        "form_name": "Pharmacokinetics (D8)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "30fabdea-700d-4f25-b303-6d321bb174a0",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "bb7c357c-6716-4cce-9c7b-96ccd4a75fd0",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b9ad15d7-344f-482b-a894-1c9476c9c218",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0f4a5bb3-2cb0-4e75-bb55-523d6ecc436d",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1fe3bf34-b688-41cd-bd61-38c2be8d5a41",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e6108d0b-038c-4a9e-be34-28949925c7d5",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8ea47020-1961-431d-a66d-be910ab5044c",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f44b9aee-0acd-4867-80e8-309aabeb4d6c",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "589a260f-0fde-4855-99b9-cd450c07bdd9",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "52697bfc-a742-4cd7-9516-3d9edf24e42a",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "af819349-cbef-4e90-a876-3527f738c616",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "57e36f41-0be5-4736-8288-8b95c1603989",
        "form_id": "",
        "form_name": "Pharmacokinetics (D9)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "VS017",
        "ecs_id": "9f00f254-1085-419d-a96f-194cf1c01f79",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS015",
        "ecs_id": "0bd16a1f-4630-4e44-887e-bedb672573f7",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS016",
        "ecs_id": "8fa45683-dca3-4b4f-833f-af145e477200",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND_LOG013",
        "ecs_id": "9f72c18b-307a-43ce-aa11-d5c29d928bb0",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND_LOG015",
        "ecs_id": "af6dd79a-ead6-4665-a3c7-31ea526467ca",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG016",
        "ecs_id": "97d79040-5f9c-40b1-a7e6-e7df436516ca",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG018",
        "ecs_id": "4660f51e-55c7-4a21-af56-671c6010b0fa",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0264b879-daa4-447d-a695-c0c0d49d93f2",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Query if standing position time is not at least 3 minutes (+59 sec window) after supine position time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0f7c71a8-52c8-4bf0-b7e3-2aa01edae5cd",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND005",
        "ecs_id": "8865576d-b8c7-4792-b230-cdc5fc7cb56e",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND007",
        "ecs_id": "982a060a-f224-4f12-8150-56a469450c96",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND006",
        "ecs_id": "189a39c0-2f22-466d-9bea-50f30e28b2e7",
        "form_name": "Vital Signs (D10)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "",
        "ecs_id": "387e4466-c5d3-412d-ab32-7b01368b63a5",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b9c25984-776c-494b-ba3f-1c3394122382",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "faa80b03-aefb-4616-93e7-9db3af9c1661",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "755dde4c-8f74-47cb-a462-a8ba1dc6c33c",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ecb548c7-4d47-4b4a-8678-c23cbbf4082b",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4377882e-f9bf-458d-82ce-5730556a5a33",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "31e49894-50a3-49a2-9315-c23e7b1aa166",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "772c42d8-ac8b-4e33-97bb-9a3f67241837",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b5bd5a55-cf43-4e94-9fe5-11e6cf2c8efe",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "26011fed-6381-48e1-aca4-62883f60cf11",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "143a6120-6da0-4d1f-8dff-0263407d48e9",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "be6777d7-23f8-46ee-9d7a-9e6b4c434a34",
        "form_id": "",
        "form_name": "Pharmacokinetics (D10)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b231f7e2-2bd3-4780-a539-13d6ec2a8be2",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f33f8ac1-fe2b-4df6-ab12-f8b0312b6b8f",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "743bee28-eaad-41a2-90e8-4b3e568d0c77",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "87049957-1025-4ba0-b91b-b43ae5d8ea1e",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cfaab5af-bab9-4c95-878b-ea461216067b",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5e58430d-2fde-424f-90ba-88e1ba70a885",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a79fa63e-cb51-412d-bd1a-06be750ba483",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "374ff22c-cddb-4a6a-8790-b642280c1503",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0a465c32-c475-42e7-ac8c-d05cf1700e41",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b71654dd-897c-4bc5-ae5f-792c1b161534",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2e53aa68-f2a5-4dde-91b7-f95ff8171d76",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7db8cc32-566a-4804-b492-8509ca559ec2",
        "form_id": "",
        "form_name": "Pharmacokinetics (D11)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "VS017",
        "ecs_id": "9f00f254-1085-419d-a96f-194cf1c01f79",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS015",
        "ecs_id": "0bd16a1f-4630-4e44-887e-bedb672573f7",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS016",
        "ecs_id": "8fa45683-dca3-4b4f-833f-af145e477200",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND_LOG013",
        "ecs_id": "9f72c18b-307a-43ce-aa11-d5c29d928bb0",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND_LOG015",
        "ecs_id": "af6dd79a-ead6-4665-a3c7-31ea526467ca",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG016",
        "ecs_id": "97d79040-5f9c-40b1-a7e6-e7df436516ca",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG018",
        "ecs_id": "4660f51e-55c7-4a21-af56-671c6010b0fa",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0264b879-daa4-447d-a695-c0c0d49d93f2",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Query if standing position time is not at least 3 minutes (+59 sec window) after supine position time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0f7c71a8-52c8-4bf0-b7e3-2aa01edae5cd",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND005",
        "ecs_id": "8865576d-b8c7-4792-b230-cdc5fc7cb56e",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND007",
        "ecs_id": "982a060a-f224-4f12-8150-56a469450c96",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND006",
        "ecs_id": "189a39c0-2f22-466d-9bea-50f30e28b2e7",
        "form_name": "Vital Signs (D12)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "23d46683-0438-4f3f-a98d-078eece91816",
        "form_name": "Electrocardiogram (D12)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "fd740ca4-c8ed-461b-9b38-f84d51e05439",
        "form_name": "Electrocardiogram (D12)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "e0ce35a0-e237-4302-a700-42f0b0ed55bb",
        "form_name": "Electrocardiogram (D12)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "ec08d46a-876b-411b-a227-b53e96510eec",
        "form_name": "Electrocardiogram (D12)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "",
        "ecs_id": "e729790c-ffdb-4baf-8934-db43c4e2dfeb",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8f9f66e5-d548-49e9-ae11-56105614762d",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b9572f3b-a727-400e-8f17-fefa9b6f5519",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "Were lab assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9d42c323-f1bf-4d74-976a-6025d974aa7b",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e4a90183-2309-49c1-8f4e-6f8617d6f5c5",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "37cf6545-5da8-4c04-b072-98c52be9e932",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6bb894db-2833-4646-afab-0d88404af3cb",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4971449d-a62c-4769-82fd-cf98d02ee481",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "Are the results clinically significant?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9e895c45-58a7-45b6-b2d1-07db941f05b2",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "If clinically significant, specify1",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e08feb72-32df-4a96-be12-456d6da82701",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "If clinically significant, specify2",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "eb7fb188-3186-4ae1-9d03-5366bd851901",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "If clinically significant, specify3",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b9d3687b-6abb-4892-bc28-f92cd36d7d66",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "If other, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c6b05fa9-ba9a-436b-85fd-4e955b4d7473",
        "form_id": "",
        "form_name": "Laboratory Test Collections D12",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2dad5d25-fecd-448c-9821-465778ba5ff1",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7e7264f1-1e17-408a-a2c0-b5d6badff2ba",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Scheduled time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "af8356ad-6b66-4a78-9a63-0f9f664f0b49",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Timepoint number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a2094347-d402-49dc-98c8-8e21ef302aac",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Reference dose",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e4f6bf63-7ff9-430e-8db7-0d3cef88ffab",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "57fba336-0d09-462b-9c78-0b74bd8383c8",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ee0a392b-df88-43be-97a0-9cacaeec9537",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "74bc9a46-1781-4902-a943-e085d0c8fda3",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6a7e2e98-ddf0-4dec-9f5c-544420e59044",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9d947bbc-491c-4c5c-969e-aaafd410c0aa",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f4ffe94c-54b9-4070-ab2f-44f6d61e8206",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7df1029b-8c2c-4eab-b77c-8b6bc96cd950",
        "form_id": "",
        "form_name": "Pharmacokinetics (D12)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCTPT",
          "PCTPTNUM",
          "PCTPTREF",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "dfb2625f-c833-4da8-a449-48b7bf046c9c",
        "form_id": "",
        "form_name": "Discharge Page",
        "form_field_value": "Date of Discharge",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a62c89f9-571b-487d-b372-18b2172db166",
        "form_id": "",
        "form_name": "Discharge Page",
        "form_field_value": "Time of Discharge",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSSTDAT",
          "DSTIM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5bebbf21-fd17-4f08-8fd6-75158cbd0467",
        "form_id": "",
        "form_name": "Phone Call",
        "form_field_value": "Was Phone Call performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PHNCALLPERF",
          "PHNCALLN",
          "PHNCALLDAT",
          "PHNCALLTIM",
          "PHSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c00fd69e-4d1b-458d-8cb5-89dcef5bfd03",
        "form_id": "",
        "form_name": "Phone Call",
        "form_field_value": "If No, provide reason",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PHNCALLPERF",
          "PHNCALLN",
          "PHNCALLDAT",
          "PHNCALLTIM",
          "PHSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9186c0e5-dad1-4eed-9604-fec23a3f920b",
        "form_id": "",
        "form_name": "Phone Call",
        "form_field_value": "Date performed",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PHNCALLPERF",
          "PHNCALLN",
          "PHNCALLDAT",
          "PHNCALLTIM",
          "PHSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "278da80d-f723-4325-b4f8-f1ccf59719ef",
        "form_id": "",
        "form_name": "Phone Call",
        "form_field_value": "Time performed",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PHNCALLPERF",
          "PHNCALLN",
          "PHNCALLDAT",
          "PHNCALLTIM",
          "PHSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "55295c23-2c51-4b4d-8844-4eedf5483d24",
        "form_id": "",
        "form_name": "Phone Call",
        "form_field_value": "Derived Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PHNCALLPERF",
          "PHNCALLN",
          "PHNCALLDAT",
          "PHNCALLTIM",
          "PHSTDTTM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "VS017",
        "ecs_id": "9f00f254-1085-419d-a96f-194cf1c01f79",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS015",
        "ecs_id": "0bd16a1f-4630-4e44-887e-bedb672573f7",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS016",
        "ecs_id": "8fa45683-dca3-4b4f-833f-af145e477200",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND_LOG013",
        "ecs_id": "9f72c18b-307a-43ce-aa11-d5c29d928bb0",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND_LOG015",
        "ecs_id": "af6dd79a-ead6-4665-a3c7-31ea526467ca",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG016",
        "ecs_id": "97d79040-5f9c-40b1-a7e6-e7df436516ca",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG018",
        "ecs_id": "4660f51e-55c7-4a21-af56-671c6010b0fa",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0264b879-daa4-447d-a695-c0c0d49d93f2",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Query if standing position time is not at least 3 minutes (+59 sec window) after supine position time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0f7c71a8-52c8-4bf0-b7e3-2aa01edae5cd",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND005",
        "ecs_id": "8865576d-b8c7-4792-b230-cdc5fc7cb56e",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND007",
        "ecs_id": "982a060a-f224-4f12-8150-56a469450c96",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND006",
        "ecs_id": "189a39c0-2f22-466d-9bea-50f30e28b2e7",
        "form_name": "Vital Signs (ET)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "23d46683-0438-4f3f-a98d-078eece91816",
        "form_name": "Electrocardiogram (ET)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "fd740ca4-c8ed-461b-9b38-f84d51e05439",
        "form_name": "Electrocardiogram (ET)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "e0ce35a0-e237-4302-a700-42f0b0ed55bb",
        "form_name": "Electrocardiogram (ET)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "ec08d46a-876b-411b-a227-b53e96510eec",
        "form_name": "Electrocardiogram (ET)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "",
        "ecs_id": "710af6f9-9588-46c4-9ac8-e727c72871bc",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4a20aa1e-b920-4ee1-8828-5409bd35b317",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "50869ac7-1aea-4e10-a4b8-8d07683b2c19",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "Were lab assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f22d62b3-a2a4-4e60-a6b1-3fbdc9552662",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "d67acb30-9871-456d-ab2a-8ba4b4754f39",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e0d7645a-01d2-4344-ae18-559955030a07",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c0002e39-d515-4b64-8508-4e30816754b5",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "3d18a06b-86df-4dd7-83f6-11e907462132",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "Are the results clinically significant?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c51e2b00-55e2-4ae2-8bac-961c595bbfd8",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "If clinically significant, specify1",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8dc2ed97-4873-4aeb-ae75-7a929362d7b6",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "If clinically significant, specify2",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1c77ee35-8155-4e29-8f51-94852acc4659",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "If clinically significant, specify3",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ae620f8a-a758-4e1b-9324-1bae381ded18",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "If other, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "dc7cda4a-9c3f-4526-8c91-63a5b3e86661",
        "form_id": "",
        "form_name": "Laboratory Test Collections (ET)",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBCAT",
          "LBSPEC",
          "LBYN",
          "LBREASND",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6b1beda2-0305-4fba-a4bc-ffa8a79b28da",
        "form_id": "",
        "form_name": "Pharmacokinetics (ET)",
        "form_field_value": "Sample type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "878da91e-6f90-49c5-a173-84149361459e",
        "form_id": "",
        "form_name": "Pharmacokinetics (ET)",
        "form_field_value": "Were PK Assessments performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "35ef24c4-65c9-492a-991c-ce7021313fc1",
        "form_id": "",
        "form_name": "Pharmacokinetics (ET)",
        "form_field_value": "If no, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "43a1ab76-e442-4e9b-8117-f6d8d9edcd02",
        "form_id": "",
        "form_name": "Pharmacokinetics (ET)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "fe574a15-3b9d-4111-9cdf-b1b092635a38",
        "form_id": "",
        "form_name": "Pharmacokinetics (ET)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0e14f1db-8234-46a6-81fb-c911b6502cc4",
        "form_id": "",
        "form_name": "Pharmacokinetics (ET)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "bb426287-0446-47fe-9f3d-1550b60cbc9a",
        "form_id": "",
        "form_name": "Pharmacokinetics (ET)",
        "form_field_value": "Comment",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8e8cc04f-9cf8-4b34-8a24-cdea358d2bd0",
        "form_id": "",
        "form_name": "Pharmacokinetics (ET)",
        "form_field_value": "Primary Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a707587a-7a22-4323-9171-96d286dcfe5b",
        "form_id": "",
        "form_name": "Pharmacokinetics (ET)",
        "form_field_value": "Backup Accession Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "PCSPEC",
          "PCYN",
          "PCREASND",
          "PCDAT",
          "PCTIM",
          "PCDTTM",
          "PCCOM",
          "PCACCESS1",
          "PCACCESS2"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c17b3426-48d1-4e02-97f0-724f5258ddfc",
        "form_id": "",
        "form_name": "Study Completion",
        "form_field_value": "Did the participant complete the study?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_2",
          "DSAENUM",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9eccff7e-80a1-43b0-8780-ae04d3c73549",
        "form_id": "",
        "form_name": "Study Completion",
        "form_field_value": "Date of study completion/early termination",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_2",
          "DSAENUM",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "228d1f39-4d44-4acb-82dd-5be7bd9b4b7f",
        "form_id": "",
        "form_name": "Study Completion",
        "form_field_value": "Study completed or reason for early termination",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_2",
          "DSAENUM",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "d61dc871-f344-4023-87fc-9ee9c86d7d3b",
        "form_id": "",
        "form_name": "Study Completion",
        "form_field_value": "If discontinued for 'AE' or",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_2",
          "DSAENUM",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "76117c27-038f-4389-b51f-e0444cc7b803",
        "form_id": "",
        "form_name": "Study Completion",
        "form_field_value": "If discontinued for 'death', enter date of death",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_2",
          "DSAENUM",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "19a56c4b-47c5-4d1d-8778-2b69f7faacfa",
        "form_id": "",
        "form_name": "Study Completion",
        "form_field_value": "If discontinued for 'protocol deviation' or 'other', Please specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "DSYN",
          "DSSTDAT",
          "DSDECOD_2",
          "DSAENUM",
          "DSDTDAT",
          "DSTERM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "MVAL_AE045",
        "ecs_id": "75544ac6-00d4-4570-bfd7-3e397ccb74bf",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESTDAT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE052",
        "ecs_id": "73f5267d-dce8-4040-984a-90b0579784b5",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AEENDAT is a future date)",
        "reasoning": "Field must not be a date in the future",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AEENDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for future date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE054",
        "ecs_id": "db436b6b-793d-4cfc-b99b-bb13720a682d",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESEV is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESEV']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE057",
        "ecs_id": "b43135de-5f30-4c97-b28f-ec0f2b725e66",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESER is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESER']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE059",
        "ecs_id": "e6719493-7e2a-4c59-8355-3cc014670bd1",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESDISAB is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESDISAB']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE062",
        "ecs_id": "aac3f83a-afd0-4d42-a11d-75faff6814fa",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESLIFE is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESLIFE']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE070",
        "ecs_id": "8c747727-442b-4857-b7ff-72fdbef851a1",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AEACN is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AEACN']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE079",
        "ecs_id": "fec31a31-df7b-49e2-85dc-3073e8728a68",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AEENDAT < AE.AESTDAT)",
        "reasoning": "end date can never be prior to start date",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AEENDAT', 'AESTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "End date is prior to start date. Please correct or clarify.",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE035",
        "ecs_id": "a673e7a0-317a-4155-b108-0b1c699d0070",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESER == \"Yes\") then (AE.AESDTH is enterable)",
        "reasoning": "If AE is serious then associated serious AE fields are valid for entry",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESER', 'AESDTH']",
        "action": "AE.AESDTH is enterable",
        "source": "Standard",
        "action_details": "<n/a>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE060",
        "ecs_id": "88f6b330-1c5a-4d17-bd9b-f50fbef29b7b",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESDTH is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESDTH']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE075",
        "ecs_id": "19a44780-a3dd-4547-8bd9-bcf957451d66",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AECONTRT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AECONTRT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE080",
        "ecs_id": "708a1328-960a-4889-a3f0-218cea37f2d9",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AEENDAT == AE.AESTDAT) AND (AE.AEENTIM < AE.AESTTIM)",
        "reasoning": "end time can never be prior to start time on the same day",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AEENDAT', 'AESTDAT', 'AEENTIM', 'AESTTIM']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "End time is prior to start time on the same day. Please correct or clarify.",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE088",
        "ecs_id": "410d6c89-a6cb-435c-85fc-430803d1f59f",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESER == \"Yes\") AND\n(\n(AE.AESCONG == \"No\") AND\n(AE.AESDISAB == \"No\") AND\n(AE.AESDTH == \"No\") AND\n(AE.AESHOSP == \"No\") AND\n(AE.AESLIFE == \"No\") AND\n(AE.AESMIE == \"No\")\n)",
        "reasoning": "if AE is serious then at least one of the asssociated serious AE fields must be \"Yes\"",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESER', 'AESCONG', 'AESDISAB', 'AESDTH', 'AESHOSP', 'AESLIFE', 'AESMIE']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "Serious is \"Yes\" but no associated serious field has been selected. Please correct or clarify.",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE036",
        "ecs_id": "d9af54cb-5ee5-4d62-b25f-4318afff4bcc",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESER == \"Yes\") then (AE.AESHOSP is enterable)",
        "reasoning": "If AE is serious then associated serious AE fields are valid for entry",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESER', 'AESHOSP']",
        "action": "AE.AESHOSP is enterable",
        "source": "Standard",
        "action_details": "<n/a>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE047",
        "ecs_id": "9831ff58-ecf7-4d80-84d0-7a6764b03e42",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESTDAT is an invalid date)",
        "reasoning": "Field must not be an invalid date such as 31Feb2025",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for invalid date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE053",
        "ecs_id": "6787f84f-4e49-4074-a5cf-c7bf86708152",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AEENDAT is an invalid date)",
        "reasoning": "Field must not be an invalid date such as 31Feb2025",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AEENDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for invalid date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE061",
        "ecs_id": "67423bf1-0907-43f6-a83a-331ce510238e",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESHOSP is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESHOSP']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE063",
        "ecs_id": "9862b684-25dc-4cb9-9f2f-68f253e372d1",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESMIE is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESMIE']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE077",
        "ecs_id": "2caca00f-6f2d-4045-8f4e-ab72ebef52a0",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AEOUT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AEOUT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE034",
        "ecs_id": "1aa784c7-2345-4273-98f8-9a4256537d31",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESER == \"Yes\") then (AE.AESDISAB is enterable)",
        "reasoning": "If AE is serious then associated serious AE fields are valid for entry",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESER', 'AESDISAB']",
        "action": "AE.AESDISAB is enterable",
        "source": "Standard",
        "action_details": "<n/a>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE038",
        "ecs_id": "2dfaef55-549a-4bba-bf2e-71d1f8655705",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESER == \"Yes\") then (AE.AESMIE is enterable)",
        "reasoning": "If AE is serious then associated serious AE fields are valid for entry",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESER', 'AESMIE']",
        "action": "AE.AESMIE is enterable",
        "source": "Standard",
        "action_details": "<n/a>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE064",
        "ecs_id": "57b61565-bd56-4558-8e25-9c62b3611231",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AEREL is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AEREL']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE033",
        "ecs_id": "83d6dc6f-ee0e-4e7a-9cca-d0940e5464b1",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESER == \"Yes\") then (AE.AESCONG is enterable)",
        "reasoning": "If AE is serious then associated serious AE fields are valid for entry",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESER', 'AESCONG']",
        "action": "AE.AESCONG is enterable",
        "source": "Standard",
        "action_details": "<n/a>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE037",
        "ecs_id": "43b5d957-e9f1-4338-a37c-63d3edd7ee88",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESER == \"Yes\") then (AE.AESLIFE is enterable)",
        "reasoning": "If AE is serious then associated serious AE fields are valid for entry",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESER', 'AESLIFE']",
        "action": "AE.AESLIFE is enterable",
        "source": "Standard",
        "action_details": "<n/a>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE041",
        "ecs_id": "9bea151b-da8e-4c0a-a908-df8f0cc5d82c",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AETERM is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AETERM']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE046",
        "ecs_id": "8465e1d9-7eed-473d-a7fe-79527cac39d5",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESTDAT is a future date)",
        "reasoning": "Field must not be a date in the future",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for future date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE051",
        "ecs_id": "fea2c07a-6958-4c57-8a1d-87e5b3d8474f",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AEENDAT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AEENDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE058",
        "ecs_id": "01cb11b7-8a60-4a9d-95e5-38d9c4464378",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AESCONG is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AESCONG']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_AE085",
        "ecs_id": "5389de54-b547-4faa-92e5-9d207eb42a08",
        "form_name": "Adverse Events",
        "validation_logic": "(AE.AEOUT == \"Fatal\") AND (AE.AESER == \"No\")",
        "reasoning": "a fatal outcome is always serious",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['AEOUT', 'AESER']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "Outcome is \"fatal\" but Serious is \"No\". Please correct or clarify.",
        "path": "nan"
      },
      {
        "validation_id": "AE021",
        "ecs_id": "df86ee0b-4ee5-4128-9011-282b73753697",
        "form_name": "Adverse Events",
        "validation_logic": "AECONTRT is Yes and CMYN = No, THEN fire query on AECONTRT (Was concomitant or additional treatment given?)",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['AECONTRT', 'CMYN']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Was concomitant or additional treatment given? Is Yes however Were any medications/therapies taken? Is No. Please review and update.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "DYN_AE_AESER_SET_VIS",
        "ecs_id": "650fd241-9f33-4346-afd9-3fbb6e40a2ad",
        "form_name": "Adverse Events",
        "validation_logic": "If AESER in Adverse Events IsEqualTo Y  then... AESER in Adverse Events IsPresent, and Set the datapoint used by AESDTH in Adverse Events to Visible (Don't Enter Empty When Not Visible), and Set the datapoint used by AESLIFE in Adverse Events to Visible (Don't Enter Empty When Not Visible), and Set the datapoint used by AESHOSP in Adverse Events to Visible (Don't Enter Empty When Not Visible), and Set the datapoint used by AESDISAB in Adverse Events to Visible (Don't Enter Empty When Not Visible), and Set the datapoint used by AESCONG in Adverse Events to Visible (Don't Enter Empty When Not Visible), and Set the datapoint used by AESMIE in Adverse Events to Visible (Don't Enter Empty When Not Visible)",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['AESER', 'AESDTH', 'AESLIFE', 'AESHOSP', 'AESDISAB', 'AESCONG', 'AESMIE']",
        "action": "Set Datapoint Visible",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00002"
      },
      {
        "validation_id": "MVAL_CM053",
        "ecs_id": "eb516854-b715-4bbf-a5b6-9fa3415089c4",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMDSTXT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMDSTXT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM059",
        "ecs_id": "c140c2f6-d1d0-427a-b955-12664d136505",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMDOSFRQ is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMDOSFRQ']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM061",
        "ecs_id": "2f5b3332-a886-4101-9591-089a7ca4b188",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMROUTE is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMROUTE']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM071",
        "ecs_id": "13aab767-f8ad-41c3-a020-da2740e2e4f2",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMENDAT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMENDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM076",
        "ecs_id": "92a39009-ea5d-479a-b385-f3a767f24a56",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMENDAT < CM.CMSTDAT)",
        "reasoning": "end date can never be prior to start date",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMENDAT', 'CMSTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "End date is prior to start date. Please correct or clarify.",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM065",
        "ecs_id": "f654051b-e8a7-4a39-aa87-fd7644105dee",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMSTDAT is a future date)",
        "reasoning": "Field must not be a date in the future",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMSTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for future date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM066",
        "ecs_id": "da7d129a-e316-41e6-bf42-c64c54b199ab",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMSTDAT is an invalid date)",
        "reasoning": "Field must not be an invalid date such as 31Feb2025",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMSTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for invalid date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM072",
        "ecs_id": "ae4a11ab-a2fc-46f3-b416-8273efe8209a",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMENDAT is a future date)",
        "reasoning": "Field must not be a date in the future",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMENDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for future date>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM074",
        "ecs_id": "b5cde3fb-486d-499b-92cc-91e64193dc82",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMENTIM is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMENTIM']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM040",
        "ecs_id": "ec6ecc5c-3468-4a57-b03c-9f111c5c2f7a",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMTRT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMTRT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM051",
        "ecs_id": "9eea4261-5e10-4882-abe4-7e748d167874",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMINDC is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMINDC']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM064",
        "ecs_id": "f623940a-be52-4f0e-8f6d-4f0b7c908579",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMSTDAT is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMSTDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM077",
        "ecs_id": "e6ddc252-f3e0-4ead-ade9-f220a44759b7",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMENDAT == CM.CMSTDAT) AND (CM.CMENTIM < CM.CMSTTIM)",
        "reasoning": "end time can never be prior to start time on the same day",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMENDAT', 'CMSTDAT', 'CMENTIM', 'CMSTTIM']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "End time is prior to start time on the same day. Please correct or clarify.",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM067",
        "ecs_id": "d9c51828-2e35-4cca-80a7-2f720a71e4ef",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMSTTIM is enterable and missing)",
        "reasoning": "Field must not be missing when enterable",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMSTTIM']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for missing data>",
        "path": "nan"
      },
      {
        "validation_id": "MVAL_CM073",
        "ecs_id": "1fe8b049-e68d-4a3f-a78c-d1e8f8b42d5b",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "(CM.CMENDAT is an invalid date)",
        "reasoning": "Field must not be an invalid date such as 31Feb2025",
        "indication": "nan",
        "molecule": "nan",
        "ta": "nan",
        "field_oids": "['CMENDAT']",
        "action": "prompt user with ACTION DETAILS",
        "source": "Standard",
        "action_details": "<query the field for invalid date>",
        "path": "nan"
      },
      {
        "validation_id": "CM004",
        "ecs_id": "271e4108-a590-4e24-b4ea-5b493658860d",
        "form_name": "Prior/Concomitant Medications",
        "validation_logic": "CMSTDAT is after CMENDAT, fire query on CMSTDAT (Start Date)",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['CMSTDAT', 'CMENDAT']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Concomitant Medications Start Date is after Concomitant Medications End Date. Please review and update.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "DYN_UNS_UPDATE_FOLDER_NAME",
        "ecs_id": "4e6384d2-772f-4a5d-9c3d-0967d5bb9544",
        "form_name": "Unscheduled Visit",
        "validation_logic": "If VISDAT in Visit Date in Unscheduled Visit IsPresent  then... update the folder name with the value of VISDAT in Visit Date in Unscheduled Visit",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VISDAT']",
        "action": "Set Datapoint",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Unscheduled Visit",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Unscheduled Visit",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Unscheduled Visit",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Unscheduled Visit",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "PE_TARGET003",
        "ecs_id": "a56b626e-f11c-4627-bc2e-7ef0a927a158",
        "form_name": "Physical Examination (Uns)",
        "validation_logic": "Flag if PE_TARGET.PEDAT is not equal to SV.VISDAT, THEN fire query on PE_TARGET.PEDAT (Exam Date)",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['PEDAT', 'VISDAT']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "The Physical Examination- Target 'Exam Date' is not equal to visit date. Please review and update or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS017",
        "ecs_id": "9f00f254-1085-419d-a96f-194cf1c01f79",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS015",
        "ecs_id": "0bd16a1f-4630-4e44-887e-bedb672573f7",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS016",
        "ecs_id": "8fa45683-dca3-4b4f-833f-af145e477200",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS.VSDAT & VS.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "693ee3cc-7097-4b07-8a67-61ca0703e75d",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "448b2762-2a71-4b13-bb79-542dbd096f79",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "75f72157-2dc1-454e-aed7-77ef7ffe6eb9",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Query if temperature value does not have the format ##.# or if result is outside of range. SOP range: 36.1-37.2.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['HR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "f41d1db2-4127-4c26-b244-59c14fae89ba",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Value should = NCS",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND_LOG013",
        "ecs_id": "9f72c18b-307a-43ce-aa11-d5c29d928bb0",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND_LOG015",
        "ecs_id": "af6dd79a-ead6-4665-a3c7-31ea526467ca",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS_STAND_LOG.VSDAT & VS_STAND_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_STAND_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_STAND_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG016",
        "ecs_id": "97d79040-5f9c-40b1-a7e6-e7df436516ca",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 3 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_LOG018",
        "ecs_id": "4660f51e-55c7-4a21-af56-671c6010b0fa",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS_LOG.VSDAT & VS_LOG.VSTIM is not prior to PC_LOG.PCDAT & PC_LOG.PCTIM (PK plasma Samples Log)  when PC_LOG.PCTPT & VS_LOG.VSTPT = 8 Hours_x000D_\nPost Dose, fire query on VS_LOG.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 1, Day 14",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0264b879-daa4-447d-a695-c0c0d49d93f2",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Query if standing position time is not at least 3 minutes (+59 sec window) after supine position time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "0f7c71a8-52c8-4bf0-b7e3-2aa01edae5cd",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Query if the supine position time is not at least 3 minutes before the collection time.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['RR']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "VS_STAND005",
        "ecs_id": "8865576d-b8c7-4792-b230-cdc5fc7cb56e",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_24_36H.PCDAT & PC_24_36H.PCTIM when PC_24_36H.PCTPT = 24 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 15",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND007",
        "ecs_id": "982a060a-f224-4f12-8150-56a469450c96",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_72H.PCDAT & PC_72H.PCTIM when PC_72H.PCTPT = \t_x000D_\n72 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 17",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "VS_STAND006",
        "ecs_id": "189a39c0-2f22-466d-9bea-50f30e28b2e7",
        "form_name": "Vital Signs (Uns)",
        "validation_logic": "Flag if VS_STAND.VSDAT & VS_STAND.VSTIM is not prior to PC_48_54H.PCDAT & PC_48_54H.PCTIM when PC_48_54H.PCTPT = 48 Hours Post Dose, fire query on VS_STAND.VSTIM (Time)_x000D_\n_x000D_\nFor Visits: Day 16",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "Neurology",
        "field_oids": "['VSDAT', 'VSTIM', 'PCDAT', 'PCTIM', 'PCTPT', 'VSTIM']",
        "action": "Open Query",
        "source": "Historical",
        "action_details": "Vital Signs have not been collected prior to PK collection. Please correct or clarify.",
        "path": "384-201-00002"
      },
      {
        "validation_id": "nan",
        "ecs_id": "5f9bbae4-35a1-4cd2-a983-9f26645197d2",
        "form_name": "Weight/BMI (Uns)",
        "validation_logic": "Item should not be selected if an end date is present for the substance use.",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['BMI']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "TEMPLATE_WEIGHT",
        "ecs_id": "1a3ea5a3-33fd-4b13-b229-0ea4c0a4d860",
        "form_name": "Weight/BMI (Uns)",
        "validation_logic": "Value should = Y",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['WEIGHT']",
        "action": "Is Greater than or Equal to 50",
        "source": "Historical",
        "action_details": "Weight is out of range, please review.",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "23d46683-0438-4f3f-a98d-078eece91816",
        "form_name": "Electrocardiogram (Uns)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "fd740ca4-c8ed-461b-9b38-f84d51e05439",
        "form_name": "Electrocardiogram (Uns)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "e0ce35a0-e237-4302-a700-42f0b0ed55bb",
        "form_name": "Electrocardiogram (Uns)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "ec08d46a-876b-411b-a227-b53e96510eec",
        "form_name": "Electrocardiogram (Uns)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "23d46683-0438-4f3f-a98d-078eece91816",
        "form_name": "Triplicate Electrocardiogram (Uns)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "fd740ca4-c8ed-461b-9b38-f84d51e05439",
        "form_name": "Triplicate Electrocardiogram (Uns)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "e0ce35a0-e237-4302-a700-42f0b0ed55bb",
        "form_name": "Triplicate Electrocardiogram (Uns)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "nan",
        "ecs_id": "ec08d46a-876b-411b-a227-b53e96510eec",
        "form_name": "Triplicate Electrocardiogram (Uns)",
        "validation_logic": "Query if male subject and value >= 451, if female subject and value >= 471",
        "reasoning": "nan",
        "indication": "Schizophrenia",
        "molecule": "SEP-380135",
        "ta": "nan",
        "field_oids": "['QRS']",
        "action": "nan",
        "source": "Historical",
        "action_details": "nan",
        "path": "384-201-00004"
      },
      {
        "validation_id": "",
        "ecs_id": "a6681ea7-1d1d-468a-8d7b-1db03aba6298",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "Repeat or Unscheduled?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b6a5b1c7-a6ae-40d8-b00c-8d3acb8a43bf",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "UNSCHEDULED",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "82005df7-496e-484b-a77b-f4e12767ebe7",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a1ff9ca5-8611-43ae-ba6d-e61423976816",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "d38bc356-20da-4e73-9869-20c30fd6b0a4",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "89531f26-cc39-42ae-a6ae-43cf8f1f6044",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8b97264d-0c4b-482f-a15a-ba83eea5da7f",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "101afcd1-dda4-4291-9f96-e0274ff94183",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "Are the results clinically significant?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "903622f8-1a0c-4cf0-b23f-34670fb660e5",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "If clinically significant, specify1",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "55900b27-528c-4104-9d36-1f296b309d78",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "If clinically significant, specify2",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "45c36935-3e93-4a22-8ee8-442abf0ae019",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "If clinically significant, specify3",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "338ffa6e-ebf9-42c3-b101-ca626b0286cf",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "If other, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "52332a1b-3fdb-4f22-9032-385a68205a7a",
        "form_id": "",
        "form_name": "Laboratory Test Collections (Uns)",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSTST1",
          "LBCSTST2",
          "LBCSTST3",
          "LBOTHSPE",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1e871cec-2364-400f-83b3-9872a386c5b9",
        "form_id": "",
        "form_name": "Laboratory Test_DOA/ABT (Uns)",
        "form_field_value": "Repeat or Unscheduled?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_2",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "27ed06ce-bcc7-4772-9ee3-8820787c0078",
        "form_id": "",
        "form_name": "Laboratory Test_DOA/ABT (Uns)",
        "form_field_value": "UNSCHEDULED",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_2",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9152a42c-c9e7-4704-b2d0-441ceb4a592b",
        "form_id": "",
        "form_name": "Laboratory Test_DOA/ABT (Uns)",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_2",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "99150067-cde9-4b5c-8593-7b071e899730",
        "form_id": "",
        "form_name": "Laboratory Test_DOA/ABT (Uns)",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_2",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "36c48d17-c9ee-4d14-83fb-96c81b65aeb5",
        "form_id": "",
        "form_name": "Laboratory Test_DOA/ABT (Uns)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_2",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "81e10900-bb9f-433f-baf1-32be43de0f4e",
        "form_id": "",
        "form_name": "Laboratory Test_DOA/ABT (Uns)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_2",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e7ae2167-2d95-4864-ae05-017c0932666d",
        "form_id": "",
        "form_name": "Laboratory Test_DOA/ABT (Uns)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_2",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cfb07741-35ca-4fd5-90b7-2e17f45d39ef",
        "form_id": "",
        "form_name": "Laboratory Test_DOA/ABT (Uns)",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_2",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a88a117f-bead-4c1f-b307-0f5bbd89981a",
        "form_id": "",
        "form_name": "Laboratory Test_PG (Uns)",
        "form_field_value": "Repeat or Unscheduled?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "41015d90-a90b-4bbf-9fe1-dae1b76071b0",
        "form_id": "",
        "form_name": "Laboratory Test_PG (Uns)",
        "form_field_value": "UNSCHEDULED",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "3166fd43-e55e-4830-9efb-5ba5f0c0ec5e",
        "form_id": "",
        "form_name": "Laboratory Test_PG (Uns)",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "8c5cb5f1-6371-424e-9d11-1203e4b62186",
        "form_id": "",
        "form_name": "Laboratory Test_PG (Uns)",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "70271b31-5aed-44a6-b5af-27b3f0983c7a",
        "form_id": "",
        "form_name": "Laboratory Test_PG (Uns)",
        "form_field_value": "Urine",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "7cc51142-a347-41a7-8a3b-9398ba024a83",
        "form_id": "",
        "form_name": "Laboratory Test_PG (Uns)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "54f63cb0-6829-4241-b02f-dabe8f42258e",
        "form_id": "",
        "form_name": "Laboratory Test_PG (Uns)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e9e2400f-6476-4963-ad8a-9e7f6aed27c9",
        "form_id": "",
        "form_name": "Laboratory Test_PG (Uns)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0df26cd7-b34f-4064-8ad8-2243694ae393",
        "form_id": "",
        "form_name": "Laboratory Test_PG (Uns)",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC_4",
          "LBCAT",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0aff44ee-6d9b-4e45-9419-676d272a6776",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "Repeat or Unscheduled?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "d0e585ec-8bc1-4e01-a016-31e690cb365b",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "UNSCHEDULED",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "41b28628-1c49-442d-a64a-879f0f24ef14",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "Test",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "9dd4d9d8-615b-4edc-b837-d1b01ea69407",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "Specimen Type",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "c10f559b-8570-4ad8-8cb8-c7f27e821c36",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "6fc9ab04-fc08-4164-8f55-e2e7f7bd9891",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "Time",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b3596cd9-5e0e-4c6f-9f6d-8907356bc4ba",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "Derived date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cee4c9ae-e6dc-45e4-94fa-b6d995bc45de",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "Are the results clinically significant?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "61bedfae-6071-41ab-a004-f250aacb1041",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "If yes, specify CS",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "b84f1dc0-1cda-49ea-9a67-f61913d31ada",
        "form_id": "",
        "form_name": "Laboratory Test Collection_FSH (Uns)",
        "form_field_value": "Repeat performed?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "LBREPEAT",
          "LBREPEAT",
          "LBCAT",
          "LBSPEC",
          "LBDAT",
          "LBTIM",
          "LBDTTM",
          "LBCLSIG",
          "LBCSSPEC",
          "LBRPT"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "2fd927b9-3545-4641-9102-3afe54e47491",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "ce6e8d30-87b8-4edb-81d3-f80476e60a11",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Onset Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a09d1e2f-bb5b-4a45-89dc-b0ba8f350def",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "End Date",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "d5c9d419-37a0-4eb8-ac03-a9929318ca4e",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Associated Event Term Category",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "860cf083-2190-48c7-8553-04cdae51618a",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Event Term (verbatim from participant)",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "a72b5da6-40a8-440d-8436-42d81cbc32d2",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Please specify other",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f115a12b-669a-4412-9aa5-409b820c6843",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Adverse Event Record Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f6b8b1cf-9bb6-4898-88da-f8dce7f37543",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Related to IMP",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "26a8aeea-6480-4a73-861b-336f79072e40",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Changes to the IMP due to the Event (i.e., dose increased/decreased/interrupted) 9",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "0a438a15-11a6-46df-ae0c-79368b26943e",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Requires Discontinuation",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "424150e5-b8e8-4fc4-b766-776aea92965a",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Does the participant have any previous history of this event?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4bb276ba-00ae-4da5-b223-0748160e22d9",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Date of most recent event",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "94c463c3-8441-4d4c-986d-750c5dff4831",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Has any treatment been prescribed for this current event?",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "1c25a9ce-3e40-4af2-aed9-7b601140fcaf",
        "form_id": "",
        "form_name": "Events Subject To Additional Monitoring",
        "form_field_value": "Narrative description of event",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "ESAMDAT",
          "ONSETDT",
          "ESAMENDT",
          "ESAMCAT",
          "ESAMCATO",
          "EVENTERM",
          "AENUM",
          "DRUGREL",
          "IPCHANGE",
          "ESAMDISC",
          "PREVHIST",
          "PREHISDT",
          "TRTEVENT",
          "EVENNARR"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "985ae2de-2c5c-460d-b38f-3e52cdb7a4c5",
        "form_id": "",
        "form_name": "AESI Worksheet",
        "form_field_value": "Adverse Event Number",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "AESIADT",
          "RASHSTDT",
          "RASHSTTM",
          "RASHDES",
          "RASHLOC",
          "RASHOTHSP",
          "RASHCOM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "508fc017-9fca-4524-a030-dd583c29c153",
        "form_id": "",
        "form_name": "AESI Worksheet",
        "form_field_value": "Date of Rash onset",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "AESIADT",
          "RASHSTDT",
          "RASHSTTM",
          "RASHDES",
          "RASHLOC",
          "RASHOTHSP",
          "RASHCOM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "5d57b1e9-ba8c-4220-bbed-2adb802851df",
        "form_id": "",
        "form_name": "AESI Worksheet",
        "form_field_value": "Time of Rash onset",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "AESIADT",
          "RASHSTDT",
          "RASHSTTM",
          "RASHDES",
          "RASHLOC",
          "RASHOTHSP",
          "RASHCOM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "da163409-6397-47ac-b5c8-b24f585c49f9",
        "form_id": "",
        "form_name": "AESI Worksheet",
        "form_field_value": "Description of Rash",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "AESIADT",
          "RASHSTDT",
          "RASHSTTM",
          "RASHDES",
          "RASHLOC",
          "RASHOTHSP",
          "RASHCOM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "01fe73b4-7611-45fc-aa79-8340d9b4fd89",
        "form_id": "",
        "form_name": "AESI Worksheet",
        "form_field_value": "Location of Rash",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "AESIADT",
          "RASHSTDT",
          "RASHSTTM",
          "RASHDES",
          "RASHLOC",
          "RASHOTHSP",
          "RASHCOM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "f4ebde5a-6593-4590-9e12-21f1e16e4267",
        "form_id": "",
        "form_name": "AESI Worksheet",
        "form_field_value": "If other, specify",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "AESIADT",
          "RASHSTDT",
          "RASHSTTM",
          "RASHDES",
          "RASHLOC",
          "RASHOTHSP",
          "RASHCOM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e0ef3519-0c8b-4ecb-9f0f-5ad73d70cef4",
        "form_id": "",
        "form_name": "AESI Worksheet",
        "form_field_value": "Comments",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "AESIADT",
          "RASHSTDT",
          "RASHSTTM",
          "RASHDES",
          "RASHLOC",
          "RASHOTHSP",
          "RASHCOM"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "cc46b85b-057e-4ad9-8944-5def356803e7",
        "form_id": "",
        "form_name": "Enrollment",
        "form_field_value": "Site ID",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "SITENUM",
          "SUBJID",
          "SUBJNAME"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "4d989980-ae9d-47c0-8c41-6a014d377a5f",
        "form_id": "",
        "form_name": "Enrollment",
        "form_field_value": "Participant ID",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "SITENUM",
          "SUBJID",
          "SUBJNAME"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      },
      {
        "validation_id": "",
        "ecs_id": "e0aff4f7-9639-4846-b16e-e0b0ddb4bc79",
        "form_id": "",
        "form_name": "Enrollment",
        "form_field_value": "Participant Number (Derived)",
        "validation_logic": "",
        "reasoning": "",
        "indication": "",
        "molecule": "",
        "ta": "",
        "field_oids": [
          "SITENUM",
          "SUBJID",
          "SUBJNAME"
        ],
        "action": "",
        "source": "LLM Generated",
        "action_details": "",
        "path": ""
      }
    ]
  },
  "timing": {
    "preProcessing": 0,
    "wait": 83,
    "execution": 15306532,
    "functionInternal": 15293607
  },
  "apiContext": {
    "serviceId": "ecs-utils",
    "endpointId": "generate-review-table",
    "serviceGeneration": "v11"
  }
}

In [10]:
generate_table_resp["response"]["result"][0].keys()

dict_keys(['validation_id', 'ecs_id', 'form_id', 'form_name', 'form_field_value', 'validation_logic', 'reasoning', 'indication', 'molecule', 'ta', 'field_oids', 'action', 'source', 'action_details', 'path'])

In [80]:
generate_table_resp["response"]["result"][0]

{'validation_id': '',
 'ecs_id': '07708d45-2511-429b-921c-37454a6ec19d',
 'form_id': '',
 'form_name': 'Date of Visit',
 'form_field_value': 'Visit date',
 'validation_logic': '',
 'reasoning': '',
 'indication': '',
 'molecule': '',
 'ta': '',
 'field_oids': ['SVSTDAT'],
 'action': '',
 'source': 'LLM Generated',
 'action_details': '',
 'path': ''}

## Generate API

In [81]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
 

rows = generate_table_resp["response"]["result"]
 
# Separate reusable and to-be-generated
reuse_rows = [r for r in rows if r["source"] in ("Standard", "Historical")]
llm_rows   = [r for r in rows if r["source"] == "LLM Generated"]
 
# group LLM rows by form_name
by_form = defaultdict(list)
for r in llm_rows:
    by_form[r["form_name"]].append(r)

# generation
generated_rows = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {
        executor.submit(
            generate_ecs_llm, 
            form_name, 
            fields,
            upload_resp["response"]["crf_file_id"], 
            "eval_harness"
        ): form_name
        
        for form_name, fields in by_form.items()
    }
    for future in as_completed(futures):
        form_name = futures[future]
        try:
            resp = future.result()
            generated_rows.extend(resp['response']['response'])
        except Exception as e:
            # decide: re-raise, log, or collect failures
            raise RuntimeError(f"Generation failed for form '{form_name}': {e}") from e

final = reuse_rows + generated_rows
final

DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:http://10.45.155.227:10001 "POST /dip/publicapi/projects/ECSGENERATION/llms/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:http://10.45.155.227:10001 "POST /dip/publicapi/projects/ECSGENERATION/llms/completions 

DEBUG:urllib3.connectionpool:http://10.45.155.227:10001 "POST /dip/publicapi/projects/ECSGENERATION/llms/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:http://10.45.155.227:10001 "POST /dip/publicapi/projects/ECSGENERATION/llms/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:http://10.45.155.227:10001 "POST /dip/publicapi/projects/ECSGENERATION/llms/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:http://10.45.155.227:10001 "POST /dip/publicapi/projects/ECSGENERATION/llms/completions HTTP/1.1" 200 None


[{'validation_id': 'MVAL_DM023',
  'ecs_id': '0c88ec50-e459-42a1-83d6-b6206a766f1c',
  'form_name': 'Demographics',
  'validation_logic': '(DM.ETHNIC is enterable and missing)',
  'reasoning': 'Field must not be missing when enterable',
  'indication': 'nan',
  'molecule': 'nan',
  'ta': 'nan',
  'field_oids': "['ETHNIC']",
  'action': 'prompt user with ACTION DETAILS',
  'source': 'Standard',
  'action_details': '<query the field for missing data>',
  'path': 'nan'},
 {'validation_id': 'MVAL_DM027',
  'ecs_id': '884e9ba7-5025-4249-b55d-43fe873651d9',
  'form_name': 'Demographics',
  'validation_logic': '(DM.COUNTRY is enterable and missing)',
  'reasoning': 'Field must not be missing when enterable',
  'indication': 'nan',
  'molecule': 'nan',
  'ta': 'nan',
  'field_oids': "['COUNTRY']",
  'action': 'prompt user with ACTION DETAILS',
  'source': 'Standard',
  'action_details': '<query the field for missing data>',
  'path': 'nan'},
 {'validation_id': 'MVAL_DM017',
  'ecs_id': '24ccde

In [79]:
generated_rows

[{'ecs_id': '07708d45-2511-429b-921c-37454a6ec19d',
  'form_id': None,
  'form_name': 'Date of Visit',
  'form_field_value': 'Visit date',
  'validation_logic': 'Field must not be blank AND must be a valid calendar date AND must not be a future date',
  'reasoning': 'Visit date is a required date field that must be complete, valid, and cannot be in the future as visits cannot occur in future dates',
  'action': 'Prompt user with ACTION DETAILS',
  'action_details': 'Visit date is missing, invalid, or is a future date. Please enter a valid past or current date.',
  'source': 'LLM Generated',
  'path': None,
  'form_domain_name': 'DV',
  'original_form_name': 'Date of Visit',
  'original_field': 'Visit date',
  'score': None},
 {'ecs_id': 'b61b2e5e-5b46-46f8-b4b9-9de5ac04c474',
  'form_id': None,
  'form_name': 'Laboratory Test Collections Screening',
  'form_field_value': 'Test',
  'validation_logic': 'Field must not be blank',
  'reasoning': 'Basic data completeness check',
  'action':

[{'validation_id': 'MVAL_DM023',
  'ecs_id': '0c88ec50-e459-42a1-83d6-b6206a766f1c',
  'form_name': 'Demographics',
  'validation_logic': '(DM.ETHNIC is enterable and missing)',
  'reasoning': 'Field must not be missing when enterable',
  'indication': 'nan',
  'molecule': 'nan',
  'ta': 'nan',
  'field_oids': "['ETHNIC']",
  'action': 'prompt user with ACTION DETAILS',
  'source': 'Standard',
  'action_details': '<query the field for missing data>',
  'path': 'nan'},
 {'validation_id': 'MVAL_DM027',
  'ecs_id': '884e9ba7-5025-4249-b55d-43fe873651d9',
  'form_name': 'Demographics',
  'validation_logic': '(DM.COUNTRY is enterable and missing)',
  'reasoning': 'Field must not be missing when enterable',
  'indication': 'nan',
  'molecule': 'nan',
  'ta': 'nan',
  'field_oids': "['COUNTRY']",
  'action': 'prompt user with ACTION DETAILS',
  'source': 'Standard',
  'action_details': '<query the field for missing data>',
  'path': 'nan'},
 {'validation_id': 'MVAL_DM017',
  'ecs_id': '24ccde

In [62]:
generated_rows

['response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'output_token',
 'response',
 'input_token',
 'out

In [19]:
from collections import defaultdict


generate_payloads = []
grouped = defaultdict(list)

for form in generate_table_resp["response"]["result"]:
    grouped[form["form_name"]].append(form)

for form_name, group in grouped.items():
    payload = {
        "form_name": form_name,
        "fields": group,
        "file_id": upload_resp["response"]["crf_file_id"],
        "user_id": "eval_harness"
    }
    
    generate_payloads.append(payload)

generate_payloads

[{'form_name': 'Date of Visit',
  'fields': [{'validation_id': '',
    'ecs_id': '07708d45-2511-429b-921c-37454a6ec19d',
    'form_id': '',
    'form_name': 'Date of Visit',
    'form_field_value': 'Visit date',
    'validation_logic': '',
    'reasoning': '',
    'indication': '',
    'molecule': '',
    'ta': '',
    'field_oids': ['SVSTDAT'],
    'action': '',
    'source': 'LLM Generated',
    'action_details': '',
    'path': ''}],
  'file_id': 'bd82861a-f084-4859-8a85-86f195ffe855',
  'user_id': 'eval_harness'},
 {'form_name': 'Informed Consent',
  'fields': [{'validation_id': '',
    'ecs_id': '2afb8dc4-fab6-4bd7-ab5e-ffce253d58e0',
    'form_id': '',
    'form_name': 'Informed Consent',
    'form_field_value': 'Informed consent obtained?',
    'validation_logic': '',
    'reasoning': '',
    'indication': '',
    'molecule': '',
    'ta': '',
    'field_oids': ['DSYN',
     'DSSTDAT',
     'DSTIM',
     'DSDTTM',
     'DSSAMND',
     'DSDECOD_3',
     'PROTOCOL',
     'DSYN',
 

In [76]:
#necessary imports 
from datetime import datetime
import traceback
import dataikuapi
import io
import json
import time
import base64
import uuid
# import from GLOBAL SHARED CODE
from utils import connection 
# imports from library 
from utilities.variables import RD_PROJECT_NAME
from utilities.variables import SECRET_NAME , TOKEN_KEY
from utilities.logging_config import logging
from utilities.llm import make_llm_call_batch, create_default_validation, parse_llm_batch_output
import boto3
import traceback
import json 


def generate_ecs_llm(form_name, fields, file_id, user_id):
    """
    This function is responsible for generating the ECS by llm based on form_nam eand field_name
    """
    client = dataiku.api_client()
    project = client.get_default_project()
#     llm_get = project.get_llm("agent:24Xsfa9R")
    llm_get = project.get_llm("agent:4NRmRXIB")
    generation_agent = llm_get.new_completion()
    payload = {
        "form_name":form_name,
        "field_name":fields,

    }
    generation_agent.with_context({
        "payload":payload      

    })
    generation_result = generation_agent.execute()
#     print(generation_result.text)
    response =  json.loads(generation_result.text)
    output_list = response['response']

    return {"response":response}


In [13]:
client = dataiku.api_client()
project = client.get_default_project()
llm_get = project.get_llm("agent:24Xsfa9R")

In [14]:
project.list_llms()

DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:http://10.45.155.227:10001 "GET /dip/publicapi/projects/ECSGENERATION/llms?purpose=GENERIC_COMPLETION HTTP/1.1" 200 None


[{'customClassificationRequiresHypothesisTemplate': False,
  'canDoNativeSentimentAnalysis': False,
  'canDoNativeEmotionAnalysis': False,
  'canGenerateCrossLanguageOutput': True,
  'handlesSystemMessage': True,
  'supportsImageInputs': False,
  'canBeFinetuned': True,
  'originalLLMIsUnreferenced': False,
  'friendlyName': 'Claude Sonnet 4.5 (connection: AWS-Bedrock-NOCACHE)',
  'friendlyNameShort': 'Claude Sonnet 4.5',
  'temperatureRange': {'min': 0.0, 'max': 1.0, 'step': 0.01},
  'topKRange': {'min': 0.0, 'max': 500.0, 'step': 1.0},
  'id': 'bedrock:AWS-Bedrock-NOCACHE:us.anthropic.claude-sonnet-4-5-20250929-v1:0',
  'type': 'BEDROCK',
  'connection': 'AWS-Bedrock-NOCACHE',
  'model': 'us.anthropic.claude-sonnet-4-5-20250929-v1:0',
  'promptDriven': True},
 {'customClassificationRequiresHypothesisTemplate': False,
  'canDoNativeSentimentAnalysis': False,
  'canDoNativeEmotionAnalysis': False,
  'canGenerateCrossLanguageOutput': True,
  'handlesSystemMessage': True,
  'supportsImag

In [15]:
# generate_resp = []

# for payload in generate_payloads:
#     resp = post_ecs(
#         endpoint=GENERATE_ENDPOINT,
#         headers={"Authorization": f"Bearer {GENERATE_API_KEY}"},
#         payload=payload,
#         timeout=300
#     )
#     generate_resp.append(resp)

In [37]:
generate_resp = []

for payload in generate_payloads[2:3]:
    resp = generate_ecs_llm(**payload)
    generate_resp.append(resp)

DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:http://10.45.155.227:10001 "POST /dip/publicapi/projects/ECSGENERATION/llms/completions HTTP/1.1" 200 None


{'form_name': 'Informed Consent',
 'fields': [{'validation_id': '',
   'ecs_id': '2afb8dc4-fab6-4bd7-ab5e-ffce253d58e0',
   'form_id': '',
   'form_name': 'Informed Consent',
   'form_field_value': 'Informed consent obtained?',
   'validation_logic': '',
   'reasoning': '',
   'indication': '',
   'molecule': '',
   'ta': '',
   'field_oids': ['DSYN',
    'DSSTDAT',
    'DSTIM',
    'DSDTTM',
    'DSSAMND',
    'DSDECOD_3',
    'PROTOCOL',
    'DSYN',
    'DSSTDAT',
    'DSTIM'],
   'action': '',
   'source': 'LLM Generated',
   'action_details': '',
   'path': ''},
  {'validation_id': '',
   'ecs_id': '6eadf2b1-b5b4-47aa-b115-99803687ddf0',
   'form_id': '',
   'form_name': 'Informed Consent',
   'form_field_value': 'Informed consent date',
   'validation_logic': '',
   'reasoning': '',
   'indication': '',
   'molecule': '',
   'ta': '',
   'field_oids': ['DSYN',
    'DSSTDAT',
    'DSTIM',
    'DSDTTM',
    'DSSAMND',
    'DSDECOD_3',
    'PROTOCOL',
    'DSYN',
    'DSSTDAT',
    '

In [47]:
def has_mixed_source(data):
    values = {d["source"] for d in data}
    return "LLM Generated" in values and len(values) > 1

In [46]:
for form in generate_payloads:
    print(all(field["source"] == 'LLM Generated' for field in form["fields"]))

True
True
False
True
False
False
False
False
False
True
True
True
True
True
True
False
False
False
True
True
True
True
True
True
False
True
True
True
False
True
True
True
False
True
True
True
False
True
True
False
False
True
True
True
True
False
False
True
True
True
False
False
False
False
False
False
False
False
True
True
True
True
True
True
True


In [31]:
len(generate_resp)

2

In [38]:
generate_resp

[{'response': {'response': [],
   'input_token': 0,
   'output_token': 0,
   'error': 'Traceback (most recent call last):\n  File "/tmp/tmp_folder_iVxyeLwM/dku_code.py", line 274, in process\n  File "/tmp/tmp_folder_iVxyeLwM/dku_code.py", line 274, in <listcomp>\nKeyError: \'form_field_value\'\n'}}]

In [32]:
generate_ecs_llm(**payload)

DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 10.45.155.227:10001
DEBUG:urllib3.connectionpool:http://10.45.155.227:10001 "POST /dip/publicapi/projects/ECSGENERATION/llms/completions HTTP/1.1" 200 None


{'response': []}

In [38]:
print(json.dumps(generate_resp, indent=2))

[
  {
    "response": {
      "error message": "error caused ue to Traceback (most recent call last):\n  File \"<string>\", line 52, in generate_ecs_llm\n  File \"/usr/lib64/python3.9/json/__init__.py\", line 346, in loads\n    return _default_decoder.decode(s)\n  File \"/usr/lib64/python3.9/json/decoder.py\", line 337, in decode\n    obj, end = self.raw_decode(s, idx=_w(s, 0).end())\n  File \"/usr/lib64/python3.9/json/decoder.py\", line 355, in raw_decode\n    raise JSONDecodeError(\"Expecting value\", s, err.value) from None\njson.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)\n"
    },
    "timing": {
      "preProcessing": 0,
      "wait": 995,
      "execution": 4037271,
      "functionInternal": 4032255
    },
    "apiContext": {
      "serviceId": "ecs-platfrom-agent",
      "endpointId": "ecs-generation",
      "serviceGeneration": "v4"
    }
  },
  {
    "response": {
      "error message": "error caused ue to Traceback (most recent call last):\n  File \"<s

In [75]:
for m in resp["response"]["steps"]:
    print(f"{m['step']}: {m['description']}")

0_input: Orchestrator Input
1_generator: Content Generator Agent
2_critique: Critique Agent
3_revision: Revision Agent
4_html_parser: HTML Parser Agent
5_post_processing: HTML Post-Processing
6_summary: Final Summary


In [76]:
resp["response"]["steps"]

[{'step': '0_input',
  'description': 'Orchestrator Input',
  'section_name': 'Protocol Lay Person Short Title',
  'section_prompt': 'Section Name: Protocol Lay Person Short Title\nType: Verbatim. Do not change any of the content.\n\nTask: Find and provide the exact and full Protocol Lay Person Short Title. \n\nOutput Format: Protocol Lay Person Short Title\n\nJust simply output the title and nothing else.'},
 {'step': '1_generator',
  'description': 'Content Generator Agent',
  'input': {'section_prompt': 'Section Name: Protocol Lay Person Short Title\nType: Verbatim. Do not change any of the content.\n\nTask: Find and provide the exact and full Protocol Lay Person Short Title. \n\nOutput Format: Protocol Lay Person Short Title\n\nJust simply output the title and nothing else.',
   'template': 'Verbatim from Protocol Synopsis. If there is no Lay Person Short Title, create an abbreviated version of the Protocol Title. Short title should be sufficiently detailed to make clear to a lay r

In [77]:
resp["response"]["synopsis_section_data"] # from input payload

'**Protocol Lay Person Short Title:**\n\nA Trial of the Efficacy and Safety of SEP-363856 in Acutely Psychotic Participants with Schizophrenia'

In [78]:
resp["response"]["response"]

'<html><!DOCTYPE html> <html lang="en"> <head>     <meta charset="UTF-8">     <meta name="viewport" content="width=device-width, initial-scale=1.0">     <title>Protocol Lay Person Short Title</title>     <style>         body {             font-family: Arial, sans-serif;             font-size: 12pt;             max-width: 800px;             margin: 0 auto;             line-height: 1.5;         }         sub, sup {             font-size: 14pt;         }     </style> </head> <body>     <p><span style="color: #800080;">A Trial of the Efficacy and Safety of SEP-363856 in Acutely Psychotic Participants with Schizophrenia</span></p> </body> </html>'

In [79]:
print(json.dumps(resp["response"], indent=2))

{
  "response": "<html><!DOCTYPE html> <html lang=\"en\"> <head>     <meta charset=\"UTF-8\">     <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">     <title>Protocol Lay Person Short Title</title>     <style>         body {             font-family: Arial, sans-serif;             font-size: 12pt;             max-width: 800px;             margin: 0 auto;             line-height: 1.5;         }         sub, sup {             font-size: 14pt;         }     </style> </head> <body>     <p><span style=\"color: #800080;\">A Trial of the Efficacy and Safety of SEP-363856 in Acutely Psychotic Participants with Schizophrenia</span></p> </body> </html>",
  "steps": [
    {
      "step": "0_input",
      "description": "Orchestrator Input",
      "section_name": "Protocol Lay Person Short Title",
      "section_prompt": "Section Name: Protocol Lay Person Short Title\nType: Verbatim. Do not change any of the content.\n\nTask: Find and provide the exact and full Protocol La